# Out-of-domain analysis of SOTA PEFT (Codet5p 770M_frozen) , fine tuned on CodeXGLUE.

In [ ]:
import os, math, warnings, gc, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          get_cosine_schedule_with_warmup)
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score,
                              matthews_corrcoef, balanced_accuracy_score,
                              confusion_matrix)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

os.environ["CUDA_VISIBLE_DEVICES"]    = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BB        = "Salesforce/codet5p-770m"
TRAIN_CSV = '/ACl/data/vul/traincodex.csv'
OUT       = "/results_ood_770m"

OOD_DATASETS = {
    'Ruby':       '/d/ruby.csv',
    'Go':         '/d/goo.csv',
    'JavaScript': '/d/js.csv',
    'PHP':        '/d/php.csv',
    'Kotlin':     '/d/kot.csv',
    'Fortran':    '/d/for.csv',
    'Java':       '/d/java.csv',
}

ML, BS, EP, LR = 256, 32, 15, 5e-5
WR, GA, SD, PT = 0.15, 2, 42, 5
RA, MP, MA     = 0.3, 0.5, 0.2
NL, LL, BTN    = 4, 3, 8
D              = 1024
NUM_WORKERS    = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU  = torch.cuda.device_count() if torch.cuda.is_available() else 0
torch.manual_seed(SD)
np.random.seed(SD)

PEFTS = ['LoRA', 'PrefixTuning', 'HadamardTuning', 'AdapterTuning',
         'BitFit', 'GateRA', 'TS_PEFT', 'Hieradapter']

CKPT_FILE = os.path.join(OUT, 'run_checkpoint.json')


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def mets(y, p, pr):
    y, p, pr = np.array(y), np.array(p), np.array(pr)
    cm = confusion_matrix(y, p)
    tn, fp, fn, tp_val = (cm.ravel() if cm.size == 4 else (0, 0, 0, 0))
    specificity = tn / (tn + fp + 1e-9)
    return {
        'Acc':    accuracy_score(y, p),
        'BalAcc': balanced_accuracy_score(y, p),
        'Pre':    precision_score(y, p, zero_division=0),
        'Rec':    recall_score(y, p, zero_division=0),
        'Spe':    specificity,
        'F1':     f1_score(y, p, zero_division=0),
        'MCC':    matthews_corrcoef(y, p),
        'AUC':    roc_auc_score(y, pr),
        'TP':     int(tp_val),
        'TN':     int(tn),
        'FP':     int(fp),
        'FN':     int(fn),
    }


def rdrop(l1, l2):
    p = F.log_softmax(l1, -1)
    q = F.log_softmax(l2, -1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def save_run_checkpoint(all_res):
    os.makedirs(OUT, exist_ok=True)
    serializable = {}
    for peft_name, r in all_res.items():
        serializable[peft_name] = {
            'te':  {k: float(v) for k, v in r['te'].items()},
            'ood': {
                lang: {k: float(v) for k, v in m.items()}
                for lang, m in r['ood'].items()
            }
        }
    with open(CKPT_FILE, 'w') as f:
        json.dump(serializable, f, indent=2)


def load_run_checkpoint():
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE, 'r') as f:
            return json.load(f)
    return {}


class DS(Dataset):
    def __init__(self, codes, labels, tok):
        self.c   = [str(x) for x in codes]
        self.l   = list(labels)
        self.tok = tok

    def __len__(self):
        return len(self.c)

    def __getitem__(self, i):
        e = self.tok(self.c[i], max_length=ML, padding='max_length',
                     truncation=True, return_tensors='pt')
        return {
            'ids':   e['input_ids'].squeeze(),
            'mask':  e['attention_mask'].squeeze(),
            'label': torch.tensor(self.l[i], dtype=torch.long)
        }


class LoRAL(nn.Module):
    def __init__(self, m, r=8, a=16):
        super().__init__()
        self.m = m
        self.s = a / r
        self.A = nn.Parameter(torch.empty(r, m.in_features))
        self.B = nn.Parameter(torch.zeros(m.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        return F.linear(x, self.m.weight, self.m.bias) + (x @ self.A.T @ self.B.T) * self.s


class LoHaL(nn.Module):
    def __init__(self, m, r=4, a=8):
        super().__init__()
        self.m  = m
        self.s  = a / r
        self.A1 = nn.Parameter(torch.empty(r, m.in_features))
        self.B1 = nn.Parameter(torch.zeros(m.out_features, r))
        self.A2 = nn.Parameter(torch.empty(r, m.in_features))
        self.B2 = nn.Parameter(torch.empty(m.out_features, r))
        nn.init.kaiming_uniform_(self.A1, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.A2, a=math.sqrt(5))
        nn.init.normal_(self.B2, std=1 / math.sqrt(r))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        dW = (self.B1 @ self.A1) * (self.B2 @ self.A2)
        return F.linear(x, self.m.weight, self.m.bias) + (x @ dW.T) * self.s


class AptL(nn.Module):
    def __init__(self, m, btn=64):
        super().__init__()
        self.m  = m
        f       = m.out_features
        self.d  = nn.Linear(f, btn)
        self.ac = nn.GELU()
        self.u  = nn.Linear(btn, f)
        nn.init.normal_(self.d.weight, std=1e-3)
        for x in [self.d.bias, self.u.weight, self.u.bias]:
            nn.init.zeros_(x)

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        h = F.linear(x, self.m.weight, self.m.bias)
        return h + self.u(self.ac(self.d(h)))


class GateL(nn.Module):
    def __init__(self, m, r=8, a=16):
        super().__init__()
        self.m = m
        self.s = a / r
        self.A = nn.Parameter(torch.empty(r, m.in_features))
        self.B = nn.Parameter(torch.zeros(m.out_features, r))
        self.g = nn.Linear(m.in_features, 1)
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.g.weight)
        nn.init.zeros_(self.g.bias)

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        return (F.linear(x, self.m.weight, self.m.bias) +
                torch.sigmoid(self.g(x)) * (x @ self.A.T @ self.B.T) * self.s)


class TSPL(nn.Module):
    def __init__(self, m, r=8, a=16, t=10.0):
        super().__init__()
        self.m   = m
        self.s   = a / r
        self.t   = t
        self.A   = nn.Parameter(torch.empty(r, m.in_features))
        self.B   = nn.Parameter(torch.zeros(m.out_features, r))
        self.tau = nn.Parameter(torch.zeros(1))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        gate = torch.sigmoid((x.norm(dim=-1, keepdim=True) - self.tau.abs()) * self.t)
        return (F.linear(x, self.m.weight, self.m.bias) +
                gate * (x @ self.A.T @ self.B.T) * self.s)


class PfxM(nn.Module):
    def __init__(self, pl, d, h=512):
        super().__init__()
        self.pl = pl
        self.t  = nn.Parameter(torch.randn(pl, d) * 0.01)
        self.r  = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, d))

    def forward(self, B, dev):
        return self.r(self.t).unsqueeze(0).expand(B, -1, -1).to(dev)


class APool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, 1)

    def forward(self, h, mask):
        s = self.w(h).squeeze(-1)
        s = s.float().masked_fill(mask.bool() == False, -1e9)
        a = torch.softmax(s, -1).to(h.dtype)
        return (h * a.unsqueeze(-1)).sum(1)


class DFuse(nn.Module):
    def __init__(self, d, n):
        super().__init__()
        self.g = nn.Linear(d, n)

    def forward(self, b, rs):
        a = torch.softmax(self.g(b), -1)
        return b + sum(a[:, i:i+1] * rs[i] for i in range(len(rs)))


class LRA(nn.Module):
    def __init__(self, d, btn):
        super().__init__()
        self.d  = nn.Linear(d, btn)
        self.a  = nn.GELU()
        self.u  = nn.Linear(btn, d)
        self.dr = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.d.weight, nonlinearity='linear')
        for x in [self.d.bias, self.u.weight, self.u.bias]:
            nn.init.zeros_(x)

    def forward(self, x):
        return self.u(self.dr(self.a(self.d(x))))


def mk_bb():
    return AutoModelForSeq2SeqLM.from_pretrained(BB)


def set_sub(m, name, mod):
    ps = name.split('.')
    p  = m
    for x in ps[:-1]:
        p = getattr(p, x)
    setattr(p, ps[-1], mod)


LTGT = ['SelfAttention.q', 'SelfAttention.v', 'query', 'value']
ATGT = ['SelfAttention.o', 'DenseReluDense.wo', 'dense', 'out_proj']


def patch(enc, make, tgts):
    for n, m in list(enc.named_modules()):
        if isinstance(m, nn.Linear) and any(t in n for t in tgts):
            set_sub(enc, n, make(m))


def clf_hd(d):
    return nn.Sequential(
        nn.Linear(d, 256), nn.GELU(), nn.Dropout(0.3),
        nn.Linear(256, 64), nn.GELU(), nn.Dropout(0.2),
        nn.Linear(64, 2)
    )


class PEFTMod(nn.Module):
    def __init__(self, peft):
        super().__init__()
        self.bb   = mk_bb()
        self.peft = peft
        self.pl   = 0
        self.pfx  = None
        for p in self.bb.parameters():
            p.requires_grad = False

        PM = {
            'LoRA':           lambda m: LoRAL(m),
            'HadamardTuning': lambda m: LoHaL(m),
            'AdapterTuning':  lambda m: AptL(m),
            'GateRA':         lambda m: GateL(m),
            'TS_PEFT':        lambda m: TSPL(m),
        }
        TM = {
            'LoRA':           LTGT,
            'HadamardTuning': LTGT,
            'AdapterTuning':  ATGT,
            'GateRA':         LTGT,
            'TS_PEFT':        LTGT,
        }
        if peft in PM:
            patch(self.bb.encoder, PM[peft], TM[peft])
        elif peft == 'BitFit':
            for n, p in self.bb.encoder.named_parameters():
                p.requires_grad = ('bias' in n)
        elif peft == 'PrefixTuning':
            self.pl  = 10
            self.pfx = PfxM(10, D)

        self.pool = APool(D)
        self.clf  = clf_hd(D)

    def encode(self, ids, mask):
        if self.peft == 'PrefixTuning':
            B    = ids.shape[0]
            s    = ML - self.pl
            ids2 = ids[:, :s]
            m2   = mask[:, :s]
            e    = self.bb.shared(ids2)
            pfx  = self.pfx(B, ids.device)
            xe   = torch.cat([pfx, e], 1)
            xm   = torch.cat([torch.ones(B, self.pl, dtype=m2.dtype, device=ids.device), m2], 1)
            h    = self.bb.encoder(inputs_embeds=xe, attention_mask=xm).last_hidden_state
            return self.pool(h[:, self.pl:], m2)
        return self.pool(
            self.bb.encoder(input_ids=ids, attention_mask=mask).last_hidden_state, mask)

    def forward(self, ids, mask):
        return self.clf(self.encode(ids, mask))


class LAPMod(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb    = mk_bb()
        for p in self.bb.parameters():
            p.requires_grad = False
        self.pools = nn.ModuleList([APool(D) for _ in range(NL + 1)])
        self.lras  = nn.ModuleList([LRA(D, BTN) for _ in range(NL)])
        self.norms = nn.ModuleList([nn.LayerNorm(D) for _ in range(NL)])
        self.fuse  = DFuse(D, NL)
        self.clf   = clf_hd(D)

    def encode(self, ids, mask):
        hs   = self.bb.encoder(input_ids=ids, attention_mask=mask,
                               output_hidden_states=True).hidden_states
        N    = len(hs)
        base = self.pools[NL](hs[-1], mask)
        lc   = [self.pools[l](hs[min((l + 1) * LL, N - 1)], mask) for l in range(NL)]
        res  = [self.norms[l](self.lras[l](lc[l] - lc[l + 1] if l < NL - 1 else lc[l]))
                for l in range(NL)]
        return self.fuse(base, res)

    def forward(self, ids, mask, emb=None):
        return self.clf(emb if emb is not None else self.encode(ids, mask))


def train_ep(model, loader, crit, opt, sched, cw, il, scaler):
    model.train()
    tl = 0
    yl, yp, ypr = [], [], []
    opt.zero_grad()
    for i, b in enumerate(loader):
        ids  = b['ids'].to(DEVICE, non_blocking=True)
        mask = b['mask'].to(DEVICE, non_blocking=True)
        lab  = b['label'].to(DEVICE, non_blocking=True)
        with autocast(device_type='cuda'):
            if il:
                emb = unwrap(model).encode(ids, mask)
                if np.random.rand() < MP:
                    lam = np.random.beta(MA, MA)
                    idx = torch.randperm(emb.size(0), device=DEVICE)
                    me  = lam * emb + (1 - lam) * emb[idx]
                    l1  = model(ids, mask, emb=me)
                    l2  = model(ids, mask, emb=me)
                    ce  = (lam * F.cross_entropy(l1, lab, weight=cw, label_smoothing=0.05) +
                           (1 - lam) * F.cross_entropy(l1, lab[idx], weight=cw, label_smoothing=0.05))
                else:
                    l1 = model(ids, mask, emb=emb)
                    l2 = model(ids, mask, emb=emb)
                    ce = crit(l1, lab)
            else:
                l1 = model(ids, mask)
                l2 = model(ids, mask)
                ce = crit(l1, lab)
            rk   = rdrop(l1, l2)
            loss = (ce + RA * rk) / GA
        scaler.scale(loss).backward()
        if (i + 1) % GA == 0 or i + 1 == len(loader):
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()
        tl += (ce + RA * rk).item()
        with torch.no_grad():
            yp.extend(l1.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(l1.float(), -1)[:, 1].cpu().numpy())
            yl.extend(lab.cpu().numpy())
    return tl / len(loader), mets(yl, yp, ypr)


def eval_ep(model, loader, crit):
    model.eval()
    tl = 0
    yl, yp, ypr = [], [], []
    with torch.no_grad():
        for b in loader:
            ids  = b['ids'].to(DEVICE, non_blocking=True)
            mask = b['mask'].to(DEVICE, non_blocking=True)
            lab  = b['label'].to(DEVICE, non_blocking=True)
            with autocast(device_type='cuda'):
                lg = model(ids, mask)
            tl += crit(lg.float(), lab).item()
            yp.extend(lg.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
            yl.extend(lab.cpu().numpy())
    return tl / len(loader), mets(yl, yp, ypr)


def infer(model, loader):
    model.eval()
    yl, yp, ypr = [], [], []
    with torch.no_grad():
        for b in loader:
            ids  = b['ids'].to(DEVICE, non_blocking=True)
            mask = b['mask'].to(DEVICE, non_blocking=True)
            with autocast(device_type='cuda'):
                lg = model(ids, mask)
            yp.extend(lg.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
            yl.extend(b['label'].numpy())
    return np.array(yl), np.array(yp), np.array(ypr)


def eval_ood(model, ood_dfs, tok):
    results = {}
    for lang, df in ood_dfs.items():
        ds = DS(df['code'].tolist(), df['label'].tolist(), tok)
        dl = DataLoader(ds, batch_size=BS, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
        yl, yp, ypr = infer(model, dl)
        results[lang] = mets(yl, yp, ypr)
        print(f"    [{lang:>12s}]  Acc={results[lang]['Acc']:.4f}  "
              f"F1={results[lang]['F1']:.4f}  AUC={results[lang]['AUC']:.4f}  "
              f"MCC={results[lang]['MCC']:.4f}  BalAcc={results[lang]['BalAcc']:.4f}")
    return results


def plot_ood_heatmap(all_res, metric, odir):
    names = list(all_res.keys())
    langs = list(OOD_DATASETS.keys())
    data  = np.array([[all_res[n]['ood'][l][metric] for l in langs] for n in names])
    fig, ax = plt.subplots(figsize=(max(10, len(langs) * 1.4), max(5, len(names) * 0.7 + 1)))
    im = ax.imshow(data, cmap='RdYlGn', aspect='auto',
                   vmin=max(0, data.min() - 0.05), vmax=min(1, data.max() + 0.05))
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(langs))); ax.set_yticks(range(len(names)))
    ax.set_xticklabels(langs, fontsize=10); ax.set_yticklabels(names, fontsize=10)
    for i in range(len(names)):
        for j in range(len(langs)):
            ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center', fontsize=8)
    ax.set_title(f'OOD {metric} per PEFT × Language', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(odir, f'ood_heatmap_{metric}.png'), dpi=120, bbox_inches='tight')
    plt.close()


def plot_ood_grouped_bars(all_res, metric, odir):
    names = list(all_res.keys())
    langs = list(OOD_DATASETS.keys())
    x     = np.arange(len(langs))
    w     = 0.8 / len(names)
    pal   = plt.cm.tab10(np.linspace(0, 1, len(names)))
    fig, ax = plt.subplots(figsize=(max(14, len(langs) * 2), 6))
    for ni, (nm, col) in enumerate(zip(names, pal)):
        vals = [all_res[nm]['ood'][l][metric] for l in langs]
        ax.bar(x + (ni - len(names) / 2 + 0.5) * w, vals, w,
               color=col, alpha=0.85, label=nm, edgecolor='k', linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels(langs, rotation=20, ha='right', fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8, ncol=len(names))
    ax.grid(axis='y', alpha=0.3)
    ax.set_title(f'OOD {metric} — All PEFT Variants Across Languages', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(odir, f'ood_bars_{metric}.png'), dpi=120, bbox_inches='tight')
    plt.close()


def plot_ood_radar(all_res, odir):
    names  = list(all_res.keys())
    langs  = list(OOD_DATASETS.keys())
    n      = len(langs)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    pal    = plt.cm.tab10(np.linspace(0, 1, len(names)))
    for metric in ['F1', 'AUC', 'Acc']:
        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
        ax.set_title(f'OOD {metric} Radar — PEFT Comparison', fontweight='bold',
                     fontsize=12, pad=25)
        for mi, (nm, col) in enumerate(zip(names, pal)):
            vals = [all_res[nm]['ood'][l][metric] for l in langs]
            vals += vals[:1]
            ax.plot(angles, vals, color=col, linewidth=2, label=nm)
            ax.fill(angles, vals, color=col, alpha=0.07)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(langs, size=9)
        ax.set_ylim(0, 1)
        ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
        plt.tight_layout()
        plt.savefig(os.path.join(odir, f'ood_radar_{metric}.png'), dpi=120, bbox_inches='tight')
        plt.close()


def plot_val_metrics(all_res, odir):
    names = list(all_res.keys())
    mk    = ['Acc', 'BalAcc', 'Pre', 'Rec', 'F1', 'MCC', 'AUC']
    x     = np.arange(len(names))
    w     = 0.1
    cs    = plt.cm.tab10(np.linspace(0, 0.9, len(mk)))
    fig, ax = plt.subplots(figsize=(max(14, len(names) * 1.8), 6))
    for j, (k, c) in enumerate(zip(mk, cs)):
        vals = []
        for n in names:
            v = all_res[n]['te'].get(k, 0.0)
            vals.append(float(v) if not isinstance(v, str) else 0.0)
        ax.bar(x + (j - len(mk) / 2 + 0.5) * w, vals, w,
               color=c, alpha=0.85, label=k, edgecolor='k', linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=25, ha='right', fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8, ncol=len(mk))
    ax.grid(axis='y', alpha=0.3)
    ax.set_title('In-Domain Validation Metrics per PEFT', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(odir, 'val_metrics.png'), dpi=120, bbox_inches='tight')
    plt.close()


def plot_confusion_matrix_grid(all_res, odir):
    names = list(all_res.keys())
    langs = list(OOD_DATASETS.keys())
    fig, axes = plt.subplots(len(names), len(langs),
                             figsize=(len(langs) * 2.2, len(names) * 2.2))
    fig.suptitle('Confusion Matrix Counts (TP/FP/FN/TN) — PEFT × OOD Language',
                 fontweight='bold', fontsize=11, y=1.01)
    for ri, nm in enumerate(names):
        for ci, lang in enumerate(langs):
            ax  = axes[ri, ci]
            m   = all_res[nm]['ood'][lang]
            cm  = np.array([[int(m['TN']), int(m['FP'])],
                            [int(m['FN']), int(m['TP'])]])
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                        cbar=False, linewidths=0.5, linecolor='gray',
                        xticklabels=['Pred 0', 'Pred 1'],
                        yticklabels=['True 0', 'True 1'])
            ax.set_title(f'{nm}\n{lang}', fontsize=6.5, fontweight='bold')
            ax.tick_params(labelsize=5.5)
    plt.tight_layout()
    plt.savefig(os.path.join(odir, 'confusion_matrix_grid.png'),
                dpi=100, bbox_inches='tight')
    plt.close()


def main():
    os.makedirs(OUT, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(BB, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token    = tok.eos_token
        tok.pad_token_id = tok.eos_token_id

    tr  = pd.read_csv(TRAIN_CSV).rename(columns={'func': 'code'})
    ood_dfs = {}
    for lang, path in OOD_DATASETS.items():
        df = pd.read_csv(path)
        ood_dfs[lang] = df[['code', 'label']].dropna().reset_index(drop=True)
        print(f"  OOD [{lang:>12s}]: {len(ood_dfs[lang])} samples  "
              f"pos={int(ood_dfs[lang]['label'].sum())}")

    tr_df, val_df = train_test_split(tr, test_size=0.2, stratify=tr['label'],
                                     random_state=SD)
    print(f"Train: {len(tr_df)}  Val: {len(val_df)}  Device: {DEVICE}  N_GPU: {N_GPU}")

    def mkds(df):
        return DS(df['code'].tolist(), df['label'].tolist(), tok)

    tr_ds  = mkds(tr_df)
    val_ds = mkds(val_df)
    tr_dl  = DataLoader(tr_ds,  batch_size=BS, shuffle=True,  num_workers=NUM_WORKERS,
                        pin_memory=True, drop_last=True)
    val_dl = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=True)

    vc   = tr_df['label'].value_counts().sort_index().values
    cwr  = torch.tensor([1 / c for c in vc], dtype=torch.float).to(DEVICE)
    cw   = cwr / cwr.sum() * 2
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)

    completed = load_run_checkpoint()
    all_res   = {}

    for peft_name, r in completed.items():
        all_res[peft_name] = r
        print(f"  [SKIP — already done from checkpoint] PEFT: {peft_name}")

    for peft in PEFTS:
        if peft in completed:
            continue

        print(f"\n{'=' * 58}\n  PEFT: {peft}\n{'=' * 58}")
        il    = (peft == 'LAPPEFT')
        model = (LAPMod() if il else PEFTMod(peft))

        if N_GPU > 1:
            model = nn.DataParallel(model, device_ids=list(range(N_GPU)))
        model = model.to(DEVICE)

        tp = [p for p in model.parameters() if p.requires_grad]
        print(f"  Trainable params: {sum(p.numel() for p in tp):,}   "
              f"GPU replicas: {N_GPU}")

        opt    = torch.optim.AdamW(tp, lr=LR, weight_decay=0.05)
        steps  = (len(tr_dl) // GA) * EP
        ws     = int(WR * steps)
        sched  = get_cosine_schedule_with_warmup(opt, ws, steps)
        scaler = GradScaler(device='cuda')

        bf  = -1.0
        bst = None
        ni  = 0

        for ep in range(1, EP + 1):
            tl, tm = train_ep(model, tr_dl, crit, opt, sched, cw, il, scaler)
            vl, vm = eval_ep(model, val_dl, crit)
            note   = ' <--' if vm['F1'] > bf else ''
            if vm['F1'] > bf:
                bf  = vm['F1']
                bst = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                ni  = 0
            else:
                ni += 1
            print(f"  Ep{ep:2d}  TrL:{tl:.4f} F1:{tm['F1']:.4f}  "
                  f"VaL:{vl:.4f} F1:{vm['F1']:.4f} AUC:{vm['AUC']:.4f}{note}")
            if ni >= PT:
                print("  Early stop")
                break

        model.load_state_dict({k: v.to(DEVICE) for k, v in bst.items()})
        _, te_m = eval_ep(model, val_dl, crit)
        print(f"  Val metrics: {' '.join(f'{k}={v:.4f}' for k, v in te_m.items() if isinstance(v, float))}")

        print(f"  OOD Evaluation:")
        ood_res = eval_ood(model, ood_dfs, tok)

        pdir = os.path.join(OUT, peft)
        os.makedirs(pdir, exist_ok=True)

        all_res[peft] = {'te': te_m, 'ood': ood_res}

        torch.save({'s': bst, 'peft': peft}, os.path.join(pdir, 'model.pt'))
        save_run_checkpoint(all_res)
        print(f"  Checkpoint saved after {peft}")

        del model, opt, sched, scaler, bst
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    print("\n  Generating plots...")
    plot_val_metrics(all_res, OUT)
    for metric in ['Acc', 'BalAcc', 'Pre', 'Rec', 'F1', 'MCC', 'AUC']:
        plot_ood_heatmap(all_res, metric, OUT)
        plot_ood_grouped_bars(all_res, metric, OUT)
    plot_ood_radar(all_res, OUT)
    plot_confusion_matrix_grid(all_res, OUT)

    rows = []
    for nm, r in all_res.items():
        for lang in OOD_DATASETS.keys():
            m = r['ood'][lang]
            row = {'PEFT': nm, 'Language': lang}
            for k in ['Acc', 'BalAcc', 'Pre', 'Rec', 'Spe', 'F1', 'MCC', 'AUC',
                      'TP', 'TN', 'FP', 'FN']:
                row[k] = round(float(m[k]), 4) if isinstance(m[k], float) else int(m[k])
            rows.append(row)
    ood_df = pd.DataFrame(rows)
    ood_df.to_csv(os.path.join(OUT, 'ood_results.csv'), index=False)

    val_rows = []
    for nm, r in all_res.items():
        row = {'PEFT': nm}
        for k in ['Acc', 'BalAcc', 'Pre', 'Rec', 'Spe', 'F1', 'MCC', 'AUC',
                  'TP', 'TN', 'FP', 'FN']:
            v = r['te'].get(k, 0.0)
            row[k] = round(float(v), 4) if isinstance(v, float) else int(v)
        val_rows.append(row)
    pd.DataFrame(val_rows).to_csv(os.path.join(OUT, 'val_results.csv'), index=False)

    print(f"\nDone  ->  {OUT}")
    print("\n  === OOD Summary (F1) ===")
    pivot = ood_df.pivot(index='PEFT', columns='Language', values='F1')
    print(pivot.to_string())


if __name__ == '__main__':
    main()

# Out-of-domain analysis of SOTA PEFT (Codet5p 220M_frozen) , fine tuned on CodeXGLUE.

In [ ]:
import os, math, warnings, gc, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, T5EncoderModel,
                          get_cosine_schedule_with_warmup)
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score,
                              matthews_corrcoef, balanced_accuracy_score,
                              confusion_matrix)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

os.environ["CUDA_VISIBLE_DEVICES"]    = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BB        = "Salesforce/codet5p-220m"
TRAIN_CSV = 'ACl/data/vul/traincodex.csv'
OUT       = "results_ood"

OOD_DATASETS = {
    'Ruby':       '//d/ruby.csv',
    'Go':         '//d/goo.csv',
    'JavaScript': '//d/js.csv',
    'PHP':        '//d/php.csv',
    'Kotlin':     '//d/kot.csv',
    'Fortran':    '//d/for.csv',
    'Java':       '//d/java.csv',
}

ML, BS, EP, LR = 256, 32, 15, 5e-5
WR, GA, SD, PT = 0.15, 2, 42, 5
RA, MP, MA     = 0.3, 0.5, 0.2
NL, LL, BTN    = 4, 3, 8
D              = 768
NUM_WORKERS    = 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU  = torch.cuda.device_count() if torch.cuda.is_available() else 0
torch.manual_seed(SD)
np.random.seed(SD)

PEFTS = ['LoRA', 'PrefixTuning', 'HadamardTuning', 'AdapterTuning',
         'BitFit', 'GateRA', 'TS_PEFT', 'Hieradapter']

CKPT_FILE = os.path.join(OUT, 'run_checkpoint.json')


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def mets(y, p, pr):
    y, p, pr = np.array(y), np.array(p), np.array(pr)
    cm = confusion_matrix(y, p)
    tn, fp, fn, tp_val = (cm.ravel() if cm.size == 4 else (0, 0, 0, 0))
    specificity = tn / (tn + fp + 1e-9)
    return {
        'Acc':      accuracy_score(y, p),
        'BalAcc':   balanced_accuracy_score(y, p),
        'Pre':      precision_score(y, p, zero_division=0),
        'Rec':      recall_score(y, p, zero_division=0),
        'Spe':      specificity,
        'F1':       f1_score(y, p, zero_division=0),
        'MCC':      matthews_corrcoef(y, p),
        'AUC':      roc_auc_score(y, pr),
        'TP':       int(tp_val),
        'TN':       int(tn),
        'FP':       int(fp),
        'FN':       int(fn),
    }


def rdrop(l1, l2):
    p = F.log_softmax(l1, -1)
    q = F.log_softmax(l2, -1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def save_run_checkpoint(all_res):
    os.makedirs(OUT, exist_ok=True)
    serializable = {}
    for peft_name, r in all_res.items():
        serializable[peft_name] = {
            'te':     {k: float(v) for k, v in r['te'].items()},
            'ood':    {
                lang: {k: float(v) for k, v in m.items()}
                for lang, m in r['ood'].items()
            }
        }
    with open(CKPT_FILE, 'w') as f:
        json.dump(serializable, f, indent=2)


def load_run_checkpoint():
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE, 'r') as f:
            return json.load(f)
    return {}


class DS(Dataset):
    def __init__(self, codes, labels, tok):
        self.c   = [str(x) for x in codes]
        self.l   = list(labels)
        self.tok = tok

    def __len__(self):
        return len(self.c)

    def __getitem__(self, i):
        e = self.tok(self.c[i], max_length=ML, padding='max_length',
                     truncation=True, return_tensors='pt')
        return {
            'ids':   e['input_ids'].squeeze(),
            'mask':  e['attention_mask'].squeeze(),
            'label': torch.tensor(self.l[i], dtype=torch.long)
        }


class LoRAL(nn.Module):
    def __init__(self, m, r=8, a=16):
        super().__init__()
        self.m = m
        self.s = a / r
        self.A = nn.Parameter(torch.empty(r, m.in_features))
        self.B = nn.Parameter(torch.zeros(m.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        return F.linear(x, self.m.weight, self.m.bias) + (x @ self.A.T @ self.B.T) * self.s


class LoHaL(nn.Module):
    def __init__(self, m, r=4, a=8):
        super().__init__()
        self.m  = m
        self.s  = a / r
        self.A1 = nn.Parameter(torch.empty(r, m.in_features))
        self.B1 = nn.Parameter(torch.zeros(m.out_features, r))
        self.A2 = nn.Parameter(torch.empty(r, m.in_features))
        self.B2 = nn.Parameter(torch.empty(m.out_features, r))
        nn.init.kaiming_uniform_(self.A1, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.A2, a=math.sqrt(5))
        nn.init.normal_(self.B2, std=1 / math.sqrt(r))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        dW = (self.B1 @ self.A1) * (self.B2 @ self.A2)
        return F.linear(x, self.m.weight, self.m.bias) + (x @ dW.T) * self.s


class AptL(nn.Module):
    def __init__(self, m, btn=64):
        super().__init__()
        self.m  = m
        f       = m.out_features
        self.d  = nn.Linear(f, btn)
        self.ac = nn.GELU()
        self.u  = nn.Linear(btn, f)
        nn.init.normal_(self.d.weight, std=1e-3)
        for x in [self.d.bias, self.u.weight, self.u.bias]:
            nn.init.zeros_(x)

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        h = F.linear(x, self.m.weight, self.m.bias)
        return h + self.u(self.ac(self.d(h)))


class GateL(nn.Module):
    def __init__(self, m, r=8, a=16):
        super().__init__()
        self.m = m
        self.s = a / r
        self.A = nn.Parameter(torch.empty(r, m.in_features))
        self.B = nn.Parameter(torch.zeros(m.out_features, r))
        self.g = nn.Linear(m.in_features, 1)
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.g.weight)
        nn.init.zeros_(self.g.bias)

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        return (F.linear(x, self.m.weight, self.m.bias) +
                torch.sigmoid(self.g(x)) * (x @ self.A.T @ self.B.T) * self.s)


class TSPL(nn.Module):
    def __init__(self, m, r=8, a=16, t=10.0):
        super().__init__()
        self.m   = m
        self.s   = a / r
        self.t   = t
        self.A   = nn.Parameter(torch.empty(r, m.in_features))
        self.B   = nn.Parameter(torch.zeros(m.out_features, r))
        self.tau = nn.Parameter(torch.zeros(1))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        gate = torch.sigmoid((x.norm(dim=-1, keepdim=True) - self.tau.abs()) * self.t)
        return (F.linear(x, self.m.weight, self.m.bias) +
                gate * (x @ self.A.T @ self.B.T) * self.s)


class PfxM(nn.Module):
    def __init__(self, pl, d, h=512):
        super().__init__()
        self.pl = pl
        self.t  = nn.Parameter(torch.randn(pl, d) * 0.01)
        self.r  = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, d))

    def forward(self, B, dev):
        return self.r(self.t).unsqueeze(0).expand(B, -1, -1).to(dev)


class APool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, 1)

    def forward(self, h, mask):
        s = self.w(h).squeeze(-1)
        s = s.float().masked_fill(mask.bool() == False, -1e9)
        a = torch.softmax(s, -1).to(h.dtype)
        return (h * a.unsqueeze(-1)).sum(1)


class DFuse(nn.Module):
    def __init__(self, d, n):
        super().__init__()
        self.g = nn.Linear(d, n)

    def forward(self, b, rs):
        a = torch.softmax(self.g(b), -1)
        return b + sum(a[:, i:i+1] * rs[i] for i in range(len(rs)))


class LRA(nn.Module):
    def __init__(self, d, btn):
        super().__init__()
        self.d  = nn.Linear(d, btn)
        self.a  = nn.GELU()
        self.u  = nn.Linear(btn, d)
        self.dr = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.d.weight, nonlinearity='linear')
        for x in [self.d.bias, self.u.weight, self.u.bias]:
            nn.init.zeros_(x)

    def forward(self, x):
        return self.u(self.dr(self.a(self.d(x))))


def mk_bb():
    return T5EncoderModel.from_pretrained(BB)


def set_sub(m, name, mod):
    ps = name.split('.')
    p  = m
    for x in ps[:-1]:
        p = getattr(p, x)
    setattr(p, ps[-1], mod)


LTGT = ['SelfAttention.q', 'SelfAttention.v', 'query', 'value']
ATGT = ['SelfAttention.o', 'DenseReluDense.wo', 'dense', 'out_proj']


def patch(bb, make, tgts):
    for n, m in list(bb.named_modules()):
        if isinstance(m, nn.Linear) and any(t in n for t in tgts):
            set_sub(bb, n, make(m))


def clf_hd(d):
    return nn.Sequential(
        nn.Linear(d, 256), nn.GELU(), nn.Dropout(0.3),
        nn.Linear(256, 64), nn.GELU(), nn.Dropout(0.2),
        nn.Linear(64, 2)
    )


class PEFTMod(nn.Module):
    def __init__(self, peft):
        super().__init__()
        self.bb   = mk_bb()
        self.peft = peft
        self.pl   = 0
        self.pfx  = None
        for p in self.bb.parameters():
            p.requires_grad = False

        PM = {
            'LoRA':           lambda m: LoRAL(m),
            'HadamardTuning': lambda m: LoHaL(m),
            'AdapterTuning':  lambda m: AptL(m),
            'GateRA':         lambda m: GateL(m),
            'TS_PEFT':        lambda m: TSPL(m),
        }
        TM = {
            'LoRA':           LTGT,
            'HadamardTuning': LTGT,
            'AdapterTuning':  ATGT,
            'GateRA':         LTGT,
            'TS_PEFT':        LTGT,
        }
        if peft in PM:
            patch(self.bb, PM[peft], TM[peft])
        elif peft == 'BitFit':
            for n, p in self.bb.named_parameters():
                p.requires_grad = ('bias' in n)
        elif peft == 'PrefixTuning':
            self.pl  = 10
            self.pfx = PfxM(10, D)

        self.pool = APool(D)
        self.clf  = clf_hd(D)

    def encode(self, ids, mask):
        if self.peft == 'PrefixTuning':
            B    = ids.shape[0]
            s    = ML - self.pl
            ids2 = ids[:, :s]
            m2   = mask[:, :s]
            e    = self.bb.get_input_embeddings()(ids2)
            pfx  = self.pfx(B, ids.device)
            xe   = torch.cat([pfx, e], 1)
            xm   = torch.cat([torch.ones(B, self.pl, dtype=m2.dtype, device=ids.device), m2], 1)
            h    = self.bb(inputs_embeds=xe, attention_mask=xm).last_hidden_state
            return self.pool(h[:, self.pl:], m2)
        return self.pool(self.bb(input_ids=ids, attention_mask=mask).last_hidden_state, mask)

    def forward(self, ids, mask):
        return self.clf(self.encode(ids, mask))


class LAPMod(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb    = mk_bb()
        for p in self.bb.parameters():
            p.requires_grad = False
        self.pools = nn.ModuleList([APool(D) for _ in range(NL + 1)])
        self.lras  = nn.ModuleList([LRA(D, BTN) for _ in range(NL)])
        self.norms = nn.ModuleList([nn.LayerNorm(D) for _ in range(NL)])
        self.fuse  = DFuse(D, NL)
        self.clf   = clf_hd(D)

    def encode(self, ids, mask):
        hs   = self.bb(input_ids=ids, attention_mask=mask,
                       output_hidden_states=True).hidden_states
        N    = len(hs)
        base = self.pools[NL](hs[-1], mask)
        lc   = [self.pools[l](hs[min((l + 1) * LL, N - 1)], mask) for l in range(NL)]
        res  = [self.norms[l](self.lras[l](lc[l] - lc[l + 1] if l < NL - 1 else lc[l]))
                for l in range(NL)]
        return self.fuse(base, res)

    def forward(self, ids, mask, emb=None):
        return self.clf(emb if emb is not None else self.encode(ids, mask))


def train_ep(model, loader, crit, opt, sched, cw, il, scaler):
    model.train()
    tl = 0
    yl, yp, ypr = [], [], []
    opt.zero_grad()
    for i, b in enumerate(loader):
        ids  = b['ids'].to(DEVICE, non_blocking=True)
        mask = b['mask'].to(DEVICE, non_blocking=True)
        lab  = b['label'].to(DEVICE, non_blocking=True)
        with autocast(device_type='cuda'):
            if il:
                emb = unwrap(model).encode(ids, mask)
                if np.random.rand() < MP:
                    lam = np.random.beta(MA, MA)
                    idx = torch.randperm(emb.size(0), device=DEVICE)
                    me  = lam * emb + (1 - lam) * emb[idx]
                    l1  = model(ids, mask, emb=me)
                    l2  = model(ids, mask, emb=me)
                    ce  = (lam * F.cross_entropy(l1, lab, weight=cw, label_smoothing=0.05) +
                           (1 - lam) * F.cross_entropy(l1, lab[idx], weight=cw, label_smoothing=0.05))
                else:
                    l1 = model(ids, mask, emb=emb)
                    l2 = model(ids, mask, emb=emb)
                    ce = crit(l1, lab)
            else:
                l1 = model(ids, mask)
                l2 = model(ids, mask)
                ce = crit(l1, lab)
            rk   = rdrop(l1, l2)
            loss = (ce + RA * rk) / GA
        scaler.scale(loss).backward()
        if (i + 1) % GA == 0 or i + 1 == len(loader):
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()
        tl += (ce + RA * rk).item()
        with torch.no_grad():
            yp.extend(l1.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(l1.float(), -1)[:, 1].cpu().numpy())
            yl.extend(lab.cpu().numpy())
    return tl / len(loader), mets(yl, yp, ypr)


def eval_ep(model, loader, crit):
    model.eval()
    tl = 0
    yl, yp, ypr = [], [], []
    with torch.no_grad():
        for b in loader:
            ids  = b['ids'].to(DEVICE, non_blocking=True)
            mask = b['mask'].to(DEVICE, non_blocking=True)
            lab  = b['label'].to(DEVICE, non_blocking=True)
            with autocast(device_type='cuda'):
                lg = model(ids, mask)
            tl += crit(lg.float(), lab).item()
            yp.extend(lg.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
            yl.extend(lab.cpu().numpy())
    return tl / len(loader), mets(yl, yp, ypr)


def infer(model, loader):
    model.eval()
    yl, yp, ypr = [], [], []
    with torch.no_grad():
        for b in loader:
            ids  = b['ids'].to(DEVICE, non_blocking=True)
            mask = b['mask'].to(DEVICE, non_blocking=True)
            with autocast(device_type='cuda'):
                lg = model(ids, mask)
            yp.extend(lg.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
            yl.extend(b['label'].numpy())
    return np.array(yl), np.array(yp), np.array(ypr)


def eval_ood(model, ood_dfs, tok):
    results = {}
    for lang, df in ood_dfs.items():
        ds = DS(df['code'].tolist(), df['label'].tolist(), tok)
        dl = DataLoader(ds, batch_size=BS, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
        yl, yp, ypr = infer(model, dl)
        results[lang] = mets(yl, yp, ypr)
        print(f"    [{lang:>12s}]  Acc={results[lang]['Acc']:.4f}  "
              f"F1={results[lang]['F1']:.4f}  AUC={results[lang]['AUC']:.4f}  "
              f"MCC={results[lang]['MCC']:.4f}  BalAcc={results[lang]['BalAcc']:.4f}")
    return results


def plot_ood_heatmap(all_res, metric, odir):
    names = list(all_res.keys())
    langs = list(OOD_DATASETS.keys())
    data  = np.array([[all_res[n]['ood'][l][metric] for l in langs] for n in names])
    fig, ax = plt.subplots(figsize=(max(10, len(langs) * 1.4), max(5, len(names) * 0.7 + 1)))
    im = ax.imshow(data, cmap='RdYlGn', aspect='auto',
                   vmin=max(0, data.min() - 0.05), vmax=min(1, data.max() + 0.05))
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(langs))); ax.set_yticks(range(len(names)))
    ax.set_xticklabels(langs, fontsize=10); ax.set_yticklabels(names, fontsize=10)
    for i in range(len(names)):
        for j in range(len(langs)):
            ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center', fontsize=8)
    ax.set_title(f'OOD {metric} per PEFT × Language', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(odir, f'ood_heatmap_{metric}.png'), dpi=120, bbox_inches='tight')
    plt.close()


def plot_ood_grouped_bars(all_res, metric, odir):
    names = list(all_res.keys())
    langs = list(OOD_DATASETS.keys())
    x     = np.arange(len(langs))
    w     = 0.8 / len(names)
    pal   = plt.cm.tab10(np.linspace(0, 1, len(names)))
    fig, ax = plt.subplots(figsize=(max(14, len(langs) * 2), 6))
    for ni, (nm, col) in enumerate(zip(names, pal)):
        vals = [all_res[nm]['ood'][l][metric] for l in langs]
        ax.bar(x + (ni - len(names) / 2 + 0.5) * w, vals, w,
               color=col, alpha=0.85, label=nm, edgecolor='k', linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels(langs, rotation=20, ha='right', fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8, ncol=len(names))
    ax.grid(axis='y', alpha=0.3)
    ax.set_title(f'OOD {metric} — All PEFT Variants Across Languages', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(odir, f'ood_bars_{metric}.png'), dpi=120, bbox_inches='tight')
    plt.close()


def plot_ood_radar(all_res, odir):
    names  = list(all_res.keys())
    langs  = list(OOD_DATASETS.keys())
    n      = len(langs)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    pal    = plt.cm.tab10(np.linspace(0, 1, len(names)))
    for metric in ['F1', 'AUC', 'Acc']:
        fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
        ax.set_title(f'OOD {metric} Radar — PEFT Comparison', fontweight='bold',
                     fontsize=12, pad=25)
        for mi, (nm, col) in enumerate(zip(names, pal)):
            vals = [all_res[nm]['ood'][l][metric] for l in langs]
            vals += vals[:1]
            ax.plot(angles, vals, color=col, linewidth=2, label=nm)
            ax.fill(angles, vals, color=col, alpha=0.07)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(langs, size=9)
        ax.set_ylim(0, 1)
        ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=8)
        plt.tight_layout()
        plt.savefig(os.path.join(odir, f'ood_radar_{metric}.png'), dpi=120, bbox_inches='tight')
        plt.close()


def plot_val_metrics(all_res, odir):
    names = list(all_res.keys())
    mk    = ['Acc', 'BalAcc', 'Pre', 'Rec', 'F1', 'MCC', 'AUC']
    x     = np.arange(len(names))
    w     = 0.1
    cs    = plt.cm.tab10(np.linspace(0, 0.9, len(mk)))
    fig, ax = plt.subplots(figsize=(max(14, len(names) * 1.8), 6))
    for j, (k, c) in enumerate(zip(mk, cs)):
        vals = []
        for n in names:
            v = all_res[n]['te'].get(k, 0.0)
            vals.append(float(v) if not isinstance(v, str) else 0.0)
        ax.bar(x + (j - len(mk) / 2 + 0.5) * w, vals, w,
               color=c, alpha=0.85, label=k, edgecolor='k', linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=25, ha='right', fontsize=10)
    ax.set_ylim(0, 1.15)
    ax.legend(fontsize=8, ncol=len(mk))
    ax.grid(axis='y', alpha=0.3)
    ax.set_title('In-Domain Validation Metrics per PEFT', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(odir, 'val_metrics.png'), dpi=120, bbox_inches='tight')
    plt.close()


def plot_confusion_matrix_grid(all_res, odir):
    names = list(all_res.keys())
    langs = list(OOD_DATASETS.keys())
    for metric_pair in [('TP', 'FP', 'FN', 'TN')]:
        fig, axes = plt.subplots(len(names), len(langs),
                                 figsize=(len(langs) * 2.2, len(names) * 2.2))
        fig.suptitle('Confusion Matrix Counts (TP/FP/FN/TN) — PEFT × OOD Language',
                     fontweight='bold', fontsize=11, y=1.01)
        for ri, nm in enumerate(names):
            for ci, lang in enumerate(langs):
                ax  = axes[ri, ci]
                m   = all_res[nm]['ood'][lang]
                cm  = np.array([[int(m['TN']), int(m['FP'])],
                                [int(m['FN']), int(m['TP'])]])
                sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                            cbar=False, linewidths=0.5, linecolor='gray',
                            xticklabels=['Pred 0', 'Pred 1'],
                            yticklabels=['True 0', 'True 1'])
                ax.set_title(f'{nm}\n{lang}', fontsize=6.5, fontweight='bold')
                ax.tick_params(labelsize=5.5)
        plt.tight_layout()
        plt.savefig(os.path.join(odir, 'confusion_matrix_grid.png'),
                    dpi=100, bbox_inches='tight')
        plt.close()


def main():
    os.makedirs(OUT, exist_ok=True)
    tok = AutoTokenizer.from_pretrained(BB, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token    = tok.eos_token
        tok.pad_token_id = tok.eos_token_id

    tr  = pd.read_csv(TRAIN_CSV).rename(columns={'func': 'code'})
    ood_dfs = {}
    for lang, path in OOD_DATASETS.items():
        df = pd.read_csv(path)
        ood_dfs[lang] = df[['code', 'label']].dropna().reset_index(drop=True)
        print(f"  OOD [{lang:>12s}]: {len(ood_dfs[lang])} samples  "
              f"pos={int(ood_dfs[lang]['label'].sum())}")

    tr_df, val_df = train_test_split(tr, test_size=0.2, stratify=tr['label'],
                                     random_state=SD)
    print(f"Train: {len(tr_df)}  Val: {len(val_df)}  Device: {DEVICE}  N_GPU: {N_GPU}")

    def mkds(df):
        return DS(df['code'].tolist(), df['label'].tolist(), tok)

    tr_ds  = mkds(tr_df)
    val_ds = mkds(val_df)
    tr_dl  = DataLoader(tr_ds,  batch_size=BS, shuffle=True,  num_workers=NUM_WORKERS,
                        pin_memory=True, drop_last=True)
    val_dl = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=True)

    vc   = tr_df['label'].value_counts().sort_index().values
    cwr  = torch.tensor([1 / c for c in vc], dtype=torch.float).to(DEVICE)
    cw   = cwr / cwr.sum() * 2
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)

    completed = load_run_checkpoint()
    all_res   = {}

    for peft_name, r in completed.items():
        all_res[peft_name] = r
        print(f"  [SKIP — already done from checkpoint] PEFT: {peft_name}")

    for peft in PEFTS:
        if peft in completed:
            continue

        print(f"\n{'=' * 58}\n  PEFT: {peft}\n{'=' * 58}")
        il    = (peft == 'LAPPEFT')
        model = (LAPMod() if il else PEFTMod(peft))

        if N_GPU > 1:
            model = nn.DataParallel(model, device_ids=list(range(N_GPU)))
        model = model.to(DEVICE)

        tp = [p for p in model.parameters() if p.requires_grad]
        print(f"  Trainable params: {sum(p.numel() for p in tp):,}   "
              f"GPU replicas: {N_GPU}")

        opt    = torch.optim.AdamW(tp, lr=LR, weight_decay=0.05)
        steps  = (len(tr_dl) // GA) * EP
        ws     = int(WR * steps)
        sched  = get_cosine_schedule_with_warmup(opt, ws, steps)
        scaler = GradScaler(device='cuda')

        bf  = -1.0
        bst = None
        ni  = 0

        for ep in range(1, EP + 1):
            tl, tm = train_ep(model, tr_dl, crit, opt, sched, cw, il, scaler)
            vl, vm = eval_ep(model, val_dl, crit)
            note   = ' <--' if vm['F1'] > bf else ''
            if vm['F1'] > bf:
                bf  = vm['F1']
                bst = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                ni  = 0
            else:
                ni += 1
            print(f"  Ep{ep:2d}  TrL:{tl:.4f} F1:{tm['F1']:.4f}  "
                  f"VaL:{vl:.4f} F1:{vm['F1']:.4f} AUC:{vm['AUC']:.4f}{note}")
            if ni >= PT:
                print("  Early stop")
                break

        model.load_state_dict({k: v.to(DEVICE) for k, v in bst.items()})
        _, te_m = eval_ep(model, val_dl, crit)
        print(f"  Val metrics: {' '.join(f'{k}={v:.4f}' for k, v in te_m.items() if isinstance(v, float))}")

        print(f"  OOD Evaluation:")
        ood_res = eval_ood(model, ood_dfs, tok)

        pdir = os.path.join(OUT, peft)
        os.makedirs(pdir, exist_ok=True)

        all_res[peft] = {'te': te_m, 'ood': ood_res}

        torch.save({'s': bst, 'peft': peft}, os.path.join(pdir, 'model.pt'))
        save_run_checkpoint(all_res)
        print(f"  Checkpoint saved after {peft}")

        del model, opt, sched, scaler, bst
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    print("\n  Generating plots...")
    plot_val_metrics(all_res, OUT)
    for metric in ['Acc', 'BalAcc', 'Pre', 'Rec', 'F1', 'MCC', 'AUC']:
        plot_ood_heatmap(all_res, metric, OUT)
        plot_ood_grouped_bars(all_res, metric, OUT)
    plot_ood_radar(all_res, OUT)
    plot_confusion_matrix_grid(all_res, OUT)

    rows = []
    for nm, r in all_res.items():
        for lang in OOD_DATASETS.keys():
            m = r['ood'][lang]
            row = {'PEFT': nm, 'Language': lang}
            for k in ['Acc', 'BalAcc', 'Pre', 'Rec', 'Spe', 'F1', 'MCC', 'AUC',
                      'TP', 'TN', 'FP', 'FN']:
                row[k] = round(float(m[k]), 4) if isinstance(m[k], float) else int(m[k])
            rows.append(row)
    ood_df = pd.DataFrame(rows)
    ood_df.to_csv(os.path.join(OUT, 'ood_results.csv'), index=False)

    val_rows = []
    for nm, r in all_res.items():
        row = {'PEFT': nm}
        for k in ['Acc', 'BalAcc', 'Pre', 'Rec', 'Spe', 'F1', 'MCC', 'AUC',
                  'TP', 'TN', 'FP', 'FN']:
            v = r['te'].get(k, 0.0)
            row[k] = round(float(v), 4) if isinstance(v, float) else int(v)
        val_rows.append(row)
    pd.DataFrame(val_rows).to_csv(os.path.join(OUT, 'val_results.csv'), index=False)

    print(f"\nDone  ->  {OUT}")
    print("\n  === OOD Summary (F1) ===")
    pivot = ood_df.pivot(index='PEFT', columns='Language', values='F1')
    print(pivot.to_string())


if __name__ == '__main__':
    main()

# SOTA PEFT with (CodeT5p-220M_Frozen) fine tuned with (COdeXGLUE), inferecne with adversarial dataset for code vulnerbaility task

In [ ]:
import os, math, warnings, gc, json
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, T5EncoderModel,
                          get_cosine_schedule_with_warmup)
from sklearn.metrics import (accuracy_score, precision_score,
                              recall_score, f1_score, roc_auc_score)
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle, Patch
import seaborn as sns
warnings.filterwarnings('ignore')

os.environ["CUDA_VISIBLE_DEVICES"]    = "0,1,2,3"
os.environ["TOKENIZERS_PARALLELISM"]  = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

BB        = "Salesforce/codet5p-220m"
TRAIN_CSV = 'l/traincodex.csv'
TEST_CSV  = 'l/testcodex.csv'
ADV_CSV   = "/adversarial_vuln_output/adversarial_vuln_WITH_LABELS.csv"
OUT       = "/results"

ML, BS, EP, LR = 256, 32, 15, 5e-5
WR, GA, SD, PT = 0.15, 2, 42, 5
RA, MP, MA     = 0.3, 0.5, 0.2
NL, LL, BTN    = 4, 3, 8
D              = 768
NUM_WORKERS    = 4
VIZ_SAMPLES    = 3
VIZ_TOK_MAX    = 48

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU  = torch.cuda.device_count() if torch.cuda.is_available() else 0
torch.manual_seed(SD)
np.random.seed(SD)

PEFTS = ['LoRA', 'PrefixTuning', 'HadamardTuning', 'AdapterTuning',
         'BitFit', 'GateRA', 'TS_PEFT', 'hieradapter']

SEC_KWS = {
    'strcpy','strncpy','memcpy','memmove','memset','sprintf','snprintf',
    'gets','fgets','scanf','sscanf','strcat','strncat','strlen',
    'free','malloc','calloc','realloc','alloc','buffer','buf',
    'ptr','size','len','count','idx','index','offset','bound',
    'write','read','recv','send','open','fopen','popen','system','exec',
    'overflow','underflow','null','nullptr','return','goto',
    'delete','new','assert','abort','exit','check','verify'
}

ADV_ATTN_STORE = {}
REFERENCE_IDX  = None

CKPT_FILE = os.path.join(OUT, 'run_checkpoint.json')


def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def is_sec_token(tok_str):
    cleaned = tok_str.lower().strip('▁Ġ<>()[]{}.,;:!@#$%^&*-+=|/\\\'"` \t\n')
    return any(kw in cleaned for kw in SEC_KWS)


def get_toks(tok, ids, mask, n=VIZ_TOK_MAX):
    t = tok.convert_ids_to_tokens(ids.tolist())
    return [t[i] for i, v in enumerate(mask.tolist()) if v][:n]


def mets(y, p, pr):
    y, p, pr = np.array(y), np.array(p), np.array(pr)
    return {
        'Acc': accuracy_score(y, p),
        'Pre': precision_score(y, p, zero_division=0),
        'Rec': recall_score(y, p, zero_division=0),
        'F1':  f1_score(y, p, zero_division=0),
        'AUC': roc_auc_score(y, pr)
    }


def rdrop(l1, l2):
    p = F.log_softmax(l1, -1)
    q = F.log_softmax(l2, -1)
    return (F.kl_div(p, q.exp(), reduction='batchmean') +
            F.kl_div(q, p.exp(), reduction='batchmean')) / 2


def save_run_checkpoint(all_res):
    os.makedirs(OUT, exist_ok=True)
    serializable = {}
    for peft_name, r in all_res.items():
        serializable[peft_name] = {
            'om':   {k: float(v) for k, v in r['om'].items()},
            'am':   {k: float(v) for k, v in r['am'].items()},
            'te':   {k: float(v) for k, v in r['te'].items()},
            'asr':  float(r['asr']),
            'rob':  float(r['rob']),
            'flip': int(r['flip'])
        }
    with open(CKPT_FILE, 'w') as f:
        json.dump(serializable, f, indent=2)


def load_run_checkpoint():
    if os.path.exists(CKPT_FILE):
        with open(CKPT_FILE, 'r') as f:
            return json.load(f)
    return {}


class DS(Dataset):
    def __init__(self, codes, labels, tok):
        self.c   = [str(x) for x in codes]
        self.l   = list(labels)
        self.tok = tok

    def __len__(self):
        return len(self.c)

    def __getitem__(self, i):
        e = self.tok(self.c[i], max_length=ML, padding='max_length',
                     truncation=True, return_tensors='pt')
        return {
            'ids':   e['input_ids'].squeeze(),
            'mask':  e['attention_mask'].squeeze(),
            'label': torch.tensor(self.l[i], dtype=torch.long)
        }


class LoRAL(nn.Module):
    def __init__(self, m, r=8, a=16):
        super().__init__()
        self.m = m
        self.s = a / r
        self.A = nn.Parameter(torch.empty(r, m.in_features))
        self.B = nn.Parameter(torch.zeros(m.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        return F.linear(x, self.m.weight, self.m.bias) + (x @ self.A.T @ self.B.T) * self.s


class LoHaL(nn.Module):
    def __init__(self, m, r=4, a=8):
        super().__init__()
        self.m  = m
        self.s  = a / r
        self.A1 = nn.Parameter(torch.empty(r, m.in_features))
        self.B1 = nn.Parameter(torch.zeros(m.out_features, r))
        self.A2 = nn.Parameter(torch.empty(r, m.in_features))
        self.B2 = nn.Parameter(torch.empty(m.out_features, r))
        nn.init.kaiming_uniform_(self.A1, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.A2, a=math.sqrt(5))
        nn.init.normal_(self.B2, std=1 / math.sqrt(r))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        dW = (self.B1 @ self.A1) * (self.B2 @ self.A2)
        return F.linear(x, self.m.weight, self.m.bias) + (x @ dW.T) * self.s


class AptL(nn.Module):
    def __init__(self, m, btn=64):
        super().__init__()
        self.m  = m
        f       = m.out_features
        self.d  = nn.Linear(f, btn)
        self.ac = nn.GELU()
        self.u  = nn.Linear(btn, f)
        nn.init.normal_(self.d.weight, std=1e-3)
        for x in [self.d.bias, self.u.weight, self.u.bias]:
            nn.init.zeros_(x)

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        h = F.linear(x, self.m.weight, self.m.bias)
        return h + self.u(self.ac(self.d(h)))


class GateL(nn.Module):
    def __init__(self, m, r=8, a=16):
        super().__init__()
        self.m = m
        self.s = a / r
        self.A = nn.Parameter(torch.empty(r, m.in_features))
        self.B = nn.Parameter(torch.zeros(m.out_features, r))
        self.g = nn.Linear(m.in_features, 1)
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.g.weight)
        nn.init.zeros_(self.g.bias)

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        return (F.linear(x, self.m.weight, self.m.bias) +
                torch.sigmoid(self.g(x)) * (x @ self.A.T @ self.B.T) * self.s)


class TSPL(nn.Module):
    def __init__(self, m, r=8, a=16, t=10.0):
        super().__init__()
        self.m   = m
        self.s   = a / r
        self.t   = t
        self.A   = nn.Parameter(torch.empty(r, m.in_features))
        self.B   = nn.Parameter(torch.zeros(m.out_features, r))
        self.tau = nn.Parameter(torch.zeros(1))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))

    @property
    def weight(self):
        return self.m.weight

    @property
    def bias(self):
        return self.m.bias

    @property
    def in_features(self):
        return self.m.in_features

    @property
    def out_features(self):
        return self.m.out_features

    def forward(self, x):
        gate = torch.sigmoid((x.norm(dim=-1, keepdim=True) - self.tau.abs()) * self.t)
        return (F.linear(x, self.m.weight, self.m.bias) +
                gate * (x @ self.A.T @ self.B.T) * self.s)


class PfxM(nn.Module):
    def __init__(self, pl, d, h=512):
        super().__init__()
        self.pl = pl
        self.t  = nn.Parameter(torch.randn(pl, d) * 0.01)
        self.r  = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, d))

    def forward(self, B, dev):
        return self.r(self.t).unsqueeze(0).expand(B, -1, -1).to(dev)


class APool(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.w = nn.Linear(d, 1)

    def forward(self, h, mask):
        s = self.w(h).squeeze(-1)
        s = s.float().masked_fill(mask.bool() == False, -1e9)
        a = torch.softmax(s, -1).to(h.dtype)
        return (h * a.unsqueeze(-1)).sum(1)


class DFuse(nn.Module):
    def __init__(self, d, n):
        super().__init__()
        self.g = nn.Linear(d, n)

    def forward(self, b, rs):
        a = torch.softmax(self.g(b), -1)
        return b + sum(a[:, i:i+1] * rs[i] for i in range(len(rs)))


class LRA(nn.Module):
    def __init__(self, d, btn):
        super().__init__()
        self.d  = nn.Linear(d, btn)
        self.a  = nn.GELU()
        self.u  = nn.Linear(btn, d)
        self.dr = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.d.weight, nonlinearity='linear')
        for x in [self.d.bias, self.u.weight, self.u.bias]:
            nn.init.zeros_(x)

    def forward(self, x):
        return self.u(self.dr(self.a(self.d(x))))


def mk_bb():
    return T5EncoderModel.from_pretrained(BB)


def set_sub(m, name, mod):
    ps = name.split('.')
    p  = m
    for x in ps[:-1]:
        p = getattr(p, x)
    setattr(p, ps[-1], mod)


LTGT = ['SelfAttention.q', 'SelfAttention.v', 'query', 'value']
ATGT = ['SelfAttention.o', 'DenseReluDense.wo', 'dense', 'out_proj']


def patch(bb, make, tgts):
    for n, m in list(bb.named_modules()):
        if isinstance(m, nn.Linear) and any(t in n for t in tgts):
            set_sub(bb, n, make(m))


def clf_hd(d):
    return nn.Sequential(
        nn.Linear(d, 256), nn.GELU(), nn.Dropout(0.3),
        nn.Linear(256, 64), nn.GELU(), nn.Dropout(0.2),
        nn.Linear(64, 2)
    )


class PEFTMod(nn.Module):
    def __init__(self, peft):
        super().__init__()
        self.bb   = mk_bb()
        self.peft = peft
        self.pl   = 0
        self.pfx  = None
        for p in self.bb.parameters():
            p.requires_grad = False

        PM = {
            'LoRA':           lambda m: LoRAL(m),
            'HadamardTuning': lambda m: LoHaL(m),
            'AdapterTuning':  lambda m: AptL(m),
            'GateRA':         lambda m: GateL(m),
            'TS_PEFT':        lambda m: TSPL(m),
        }
        TM = {
            'LoRA':           LTGT,
            'HadamardTuning': LTGT,
            'AdapterTuning':  ATGT,
            'GateRA':         LTGT,
            'TS_PEFT':        LTGT,
        }
        if peft in PM:
            patch(self.bb, PM[peft], TM[peft])
        elif peft == 'BitFit':
            for n, p in self.bb.named_parameters():
                p.requires_grad = ('bias' in n)
        elif peft == 'PrefixTuning':
            self.pl  = 10
            self.pfx = PfxM(10, D)

        self.pool = APool(D)
        self.clf  = clf_hd(D)

    def encode(self, ids, mask):
        if self.peft == 'PrefixTuning':
            B    = ids.shape[0]
            s    = ML - self.pl
            ids2 = ids[:, :s]
            m2   = mask[:, :s]
            e    = self.bb.get_input_embeddings()(ids2)
            pfx  = self.pfx(B, ids.device)
            xe   = torch.cat([pfx, e], 1)
            xm   = torch.cat([torch.ones(B, self.pl, dtype=m2.dtype, device=ids.device), m2], 1)
            h    = self.bb(inputs_embeds=xe, attention_mask=xm).last_hidden_state
            return self.pool(h[:, self.pl:], m2)
        return self.pool(self.bb(input_ids=ids, attention_mask=mask).last_hidden_state, mask)

    def forward(self, ids, mask):
        return self.clf(self.encode(ids, mask))


class LAPMod(nn.Module):
    def __init__(self):
        super().__init__()
        self.bb    = mk_bb()
        for p in self.bb.parameters():
            p.requires_grad = False
        self.pools = nn.ModuleList([APool(D) for _ in range(NL + 1)])
        self.lras  = nn.ModuleList([LRA(D, BTN) for _ in range(NL)])
        self.norms = nn.ModuleList([nn.LayerNorm(D) for _ in range(NL)])
        self.fuse  = DFuse(D, NL)
        self.clf   = clf_hd(D)

    def encode(self, ids, mask):
        hs   = self.bb(input_ids=ids, attention_mask=mask,
                       output_hidden_states=True).hidden_states
        N    = len(hs)
        base = self.pools[NL](hs[-1], mask)
        lc   = [self.pools[l](hs[min((l + 1) * LL, N - 1)], mask) for l in range(NL)]
        res  = [self.norms[l](self.lras[l](lc[l] - lc[l + 1] if l < NL - 1 else lc[l]))
                for l in range(NL)]
        return self.fuse(base, res)

    def forward(self, ids, mask, emb=None):
        return self.clf(emb if emb is not None else self.encode(ids, mask))


def train_ep(model, loader, crit, opt, sched, cw, il, scaler):
    model.train()
    tl = 0
    yl, yp, ypr = [], [], []
    opt.zero_grad()

    for i, b in enumerate(loader):
        ids  = b['ids'].to(DEVICE, non_blocking=True)
        mask = b['mask'].to(DEVICE, non_blocking=True)
        lab  = b['label'].to(DEVICE, non_blocking=True)

        with autocast(device_type='cuda'):
            if il:
                emb = unwrap(model).encode(ids, mask)
                if np.random.rand() < MP:
                    lam = np.random.beta(MA, MA)
                    idx = torch.randperm(emb.size(0), device=DEVICE)
                    me  = lam * emb + (1 - lam) * emb[idx]
                    l1  = model(ids, mask, emb=me)
                    l2  = model(ids, mask, emb=me)
                    ce  = (lam * F.cross_entropy(l1, lab, weight=cw, label_smoothing=0.05) +
                           (1 - lam) * F.cross_entropy(l1, lab[idx], weight=cw, label_smoothing=0.05))
                else:
                    l1 = model(ids, mask, emb=emb)
                    l2 = model(ids, mask, emb=emb)
                    ce = crit(l1, lab)
            else:
                l1 = model(ids, mask)
                l2 = model(ids, mask)
                ce = crit(l1, lab)

            rk   = rdrop(l1, l2)
            loss = (ce + RA * rk) / GA

        scaler.scale(loss).backward()

        if (i + 1) % GA == 0 or i + 1 == len(loader):
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(
                [p for p in model.parameters() if p.requires_grad], 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
            opt.zero_grad()

        tl += (ce + RA * rk).item()
        with torch.no_grad():
            yp.extend(l1.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(l1.float(), -1)[:, 1].cpu().numpy())
            yl.extend(lab.cpu().numpy())

    return tl / len(loader), mets(yl, yp, ypr)


def eval_ep(model, loader, crit):
    model.eval()
    tl = 0
    yl, yp, ypr = [], [], []
    with torch.no_grad():
        for b in loader:
            ids  = b['ids'].to(DEVICE, non_blocking=True)
            mask = b['mask'].to(DEVICE, non_blocking=True)
            lab  = b['label'].to(DEVICE, non_blocking=True)
            with autocast(device_type='cuda'):
                lg = model(ids, mask)
            tl += crit(lg.float(), lab).item()
            yp.extend(lg.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
            yl.extend(lab.cpu().numpy())
    return tl / len(loader), mets(yl, yp, ypr)


def infer(model, loader):
    model.eval()
    yl, yp, ypr = [], [], []
    with torch.no_grad():
        for b in loader:
            ids  = b['ids'].to(DEVICE, non_blocking=True)
            mask = b['mask'].to(DEVICE, non_blocking=True)
            with autocast(device_type='cuda'):
                lg = model(ids, mask)
            yp.extend(lg.argmax(-1).cpu().numpy())
            ypr.extend(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy())
            yl.extend(b['label'].numpy())
    return np.array(yl), np.array(yp), np.array(ypr)


def extract_attention(model, ids, mask, peft_name):
    m     = unwrap(model)
    ids_  = ids.unsqueeze(0).to(DEVICE)
    mask_ = mask.unsqueeze(0).to(DEVICE)
    n     = int(mask.sum().item())
    result = {'n': n, 'peft': peft_name}

    try:
        with torch.no_grad():
            if peft_name == 'LAPPEFT':
                out  = m.bb(input_ids=ids_, attention_mask=mask_,
                            output_hidden_states=True, output_attentions=True)
                hs   = out.hidden_states
                N    = len(hs)
                lp   = []
                for l in range(NL):
                    h  = hs[min((l + 1) * LL, N - 1)]
                    s  = m.pools[l].w(h).squeeze(-1).float()
                    s  = s.masked_fill(mask_.bool() == False, -1e9)
                    w  = torch.softmax(s, -1)[0].cpu().numpy()[:n]
                    lp.append(w)
                bb_attn  = out.attentions[-1][0].float().mean(0).cpu().numpy()[:n, :n]
                agg_pool = np.mean(np.stack(lp), axis=0)
                result.update({'level_pool': lp, 'agg_pool': agg_pool, 'bb_attn': bb_attn})

            elif peft_name == 'PrefixTuning':
                B    = ids_.shape[0]
                s    = ML - m.pl
                ids2 = ids_[:, :s]
                m2   = mask_[:, :s]
                e    = m.bb.get_input_embeddings()(ids2)
                pfx  = m.pfx(B, ids_.device)
                xe   = torch.cat([pfx, e], 1)
                xm   = torch.cat([torch.ones(B, m.pl, dtype=m2.dtype, device=ids_.device), m2], 1)
                out  = m.bb(inputs_embeds=xe, attention_mask=xm,
                            output_hidden_states=True, output_attentions=True)
                h_code  = out.last_hidden_state[:, m.pl:, :]
                sc      = m.pool.w(h_code).squeeze(-1).float()
                sc      = sc.masked_fill(m2.bool() == False, -1e9)
                pool_w  = torch.softmax(sc, -1)[0].cpu().numpy()[:n]
                bb_full = out.attentions[-1][0].float().mean(0).cpu().numpy()
                bb_attn = bb_full[m.pl:, m.pl:][:n, :n]
                result.update({'agg_pool': pool_w, 'bb_attn': bb_attn})

            else:
                out     = m.bb(input_ids=ids_, attention_mask=mask_,
                               output_hidden_states=True, output_attentions=True)
                h       = out.last_hidden_state
                sc      = m.pool.w(h).squeeze(-1).float()
                sc      = sc.masked_fill(mask_.bool() == False, -1e9)
                pool_w  = torch.softmax(sc, -1)[0].cpu().numpy()[:n]
                bb_attn = out.attentions[-1][0].float().mean(0).cpu().numpy()[:n, :n]
                result.update({'agg_pool': pool_w, 'bb_attn': bb_attn})

    except Exception as e:
        print(f"    [attn-extract error | {peft_name}]: {e}")

    return result


def _sec_colors(toks):
    return ['#c0392b' if is_sec_token(t) else '#2980b9' for t in toks]


def _pool_bar(ax, toks, pool_w, title):
    n   = len(toks)
    xs  = np.arange(n)
    mx  = pool_w[:n].max() + 1e-9
    ax.bar(xs, pool_w[:n] / mx, color=_sec_colors(toks),
           alpha=0.85, edgecolor='black', linewidth=0.25)
    ax.set_xticks(xs)
    ax.set_xticklabels(toks, rotation=80, ha='right', fontsize=4.5)
    ax.set_ylabel('Norm. Pool Attention', fontsize=7)
    ax.set_title(title, fontsize=8, fontweight='bold')
    ax.set_ylim(0, 1.3)
    ax.grid(axis='y', alpha=0.25, linewidth=0.4)
    ax.legend(handles=[Patch(fc='#c0392b', label='Security token'),
                       Patch(fc='#2980b9', label='Other token')],
              fontsize=5.5, loc='upper right')


def _heatmap(ax, toks, bb_attn, title):
    n = len(toks)
    a = bb_attn[:n, :n]
    sns.heatmap(a, xticklabels=toks, yticklabels=toks,
                cmap='YlOrRd', ax=ax, cbar_kws={'shrink': 0.55},
                linewidths=0, rasterized=True)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=80, ha='right', fontsize=3.8)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=3.8)
    ax.set_title(title, fontsize=8, fontweight='bold')
    for j, tok in enumerate(toks):
        if is_sec_token(tok):
            ax.add_patch(Rectangle((j, 0), 1, n, fill=False,
                                   edgecolor='#c0392b', linewidth=1.2, alpha=0.7))
            ax.add_patch(Rectangle((0, j), n, 1, fill=False,
                                   edgecolor='#c0392b', linewidth=1.2, alpha=0.7))


def plot_adv_token_study(orig_toks, adv_toks, orig_info, adv_info,
                         orig_pred, adv_pred, true_label,
                         peft_name, sample_idx, odir):
    is_lap = (peft_name == 'LAPPEFT')
    n_rows = 4 if is_lap else 3
    fig    = plt.figure(figsize=(24, n_rows * 4.8))
    gs     = gridspec.GridSpec(n_rows, 2, figure=fig, hspace=0.62, wspace=0.38)

    case   = ('FOOLED' if orig_pred == true_label and adv_pred != true_label else
              'ROBUST' if orig_pred == true_label and adv_pred == true_label else 'WRONG')
    cc     = {'FOOLED': '#c0392b', 'ROBUST': '#27ae60', 'WRONG': '#e67e22'}
    lbl    = lambda p: f"{'Vuln' if p == 1 else 'Clean'}"
    true_s = f"{'Vuln' if true_label == 1 else 'Clean'}"

    fig.suptitle(
        f"Adversarial Token Attention Analysis  ·  {peft_name}  ·  Sample #{sample_idx}\n"
        f"True: {true_s}   Orig-pred: {lbl(orig_pred)}   Adv-pred: {lbl(adv_pred)}   "
        f"Case: {case}",
        fontsize=11, fontweight='bold', color=cc.get(case, 'black'), y=1.005
    )

    o_pool = orig_info.get('agg_pool', np.zeros(len(orig_toks)))
    a_pool = adv_info.get('agg_pool', np.zeros(len(adv_toks)))
    o_bb   = orig_info.get('bb_attn', np.zeros((2, 2)))
    a_bb   = adv_info.get('bb_attn', np.zeros((2, 2)))

    ax00 = fig.add_subplot(gs[0, 0])
    ax01 = fig.add_subplot(gs[0, 1])
    _pool_bar(ax00, orig_toks, o_pool,
              f'Original Code — Pool Attention  [pred: {lbl(orig_pred)}]')
    _pool_bar(ax01, adv_toks, a_pool,
              f'Adversarial Code — Pool Attention  [pred: {lbl(adv_pred)}]')

    ax10 = fig.add_subplot(gs[1, 0])
    ax11 = fig.add_subplot(gs[1, 1])
    _heatmap(ax10, orig_toks, o_bb,
             'Original — Backbone Attention\n(red border = security-relevant token)')
    _heatmap(ax11, adv_toks, a_bb,
             'Adversarial — Backbone Attention\n(red border = security-relevant token)')

    ax20 = fig.add_subplot(gs[2, 0])
    ax21 = fig.add_subplot(gs[2, 1])

    dl  = min(len(orig_toks), len(adv_toks))
    o_n = o_pool[:dl] / (o_pool[:dl].max() + 1e-9)
    a_n = a_pool[:dl] / (a_pool[:dl].max() + 1e-9)
    diff = o_n - a_n
    d_colors = ['#c0392b' if d > 0 else '#27ae60' for d in diff]
    ax20.bar(np.arange(dl), diff, color=d_colors, alpha=0.85, edgecolor='black', linewidth=0.25)
    ax20.set_xticks(np.arange(dl))
    ax20.set_xticklabels(orig_toks[:dl], rotation=80, ha='right', fontsize=4.5)
    ax20.axhline(0, color='black', linewidth=0.7, linestyle='--')
    ax20.set_title(
        'Attention Shift  (Orig - Adv, normalised)\n'
        'Red = model loses focus on token under attack  |  '
        'Green = attention increases under attack',
        fontsize=8, fontweight='bold'
    )
    ax20.grid(axis='y', alpha=0.25, linewidth=0.4)

    sec_orig = {t: float(o_pool[i]) for i, t in enumerate(orig_toks) if is_sec_token(t)}
    sec_adv  = {t: float(a_pool[i]) for i, t in enumerate(adv_toks)  if is_sec_token(t)}
    all_sec  = sorted(set(list(sec_orig.keys()) + list(sec_adv.keys())))
    if all_sec:
        ov = np.array([sec_orig.get(t, 0) for t in all_sec])
        av = np.array([sec_adv.get(t, 0)  for t in all_sec])
        mx = max(ov.max(), av.max()) + 1e-9
        ov /= mx; av /= mx
        x  = np.arange(len(all_sec))
        w  = 0.36
        ax21.bar(x - w / 2, ov, w, color='#2980b9', alpha=0.85, label='Original',
                 edgecolor='k', linewidth=0.25)
        ax21.bar(x + w / 2, av, w, color='#c0392b', alpha=0.85, label='Adversarial',
                 edgecolor='k', linewidth=0.25)
        ax21.set_xticks(x)
        ax21.set_xticklabels(all_sec, rotation=45, ha='right', fontsize=6.5)
        ax21.set_title(
            'Security-Relevant Token Attention:\nOriginal vs Adversarial (normalised)\n'
            '<- Drop here = model is "distracted" by adversarial perturbation',
            fontsize=8, fontweight='bold'
        )
        ax21.legend(fontsize=7)
        ax21.grid(axis='y', alpha=0.25)
    else:
        ax21.text(0.5, 0.5, 'No security keywords in top tokens',
                  ha='center', va='center', transform=ax21.transAxes, fontsize=10)
        ax21.set_title('Security Token Attention', fontsize=8, fontweight='bold')

    if is_lap and 'level_pool' in orig_info:
        axL    = fig.add_subplot(gs[3, :])
        levels = orig_info['level_pool']
        n_lev  = len(levels)
        n_t    = len(orig_toks)
        xs     = np.arange(n_t)
        w_per  = 0.75 / n_lev
        cmap   = plt.cm.tab10(np.linspace(0, 0.7, n_lev))
        for li, (lw, col) in enumerate(zip(levels, cmap)):
            norm_lw = lw[:n_t] / (lw[:n_t].max() + 1e-9)
            offset  = (li - n_lev / 2 + 0.5) * w_per
            axL.bar(xs + offset, norm_lw, w_per, color=col, alpha=0.82,
                    label=f'Level {li + 1}', edgecolor='k', linewidth=0.15)
        for j, t in enumerate(orig_toks):
            if is_sec_token(t):
                axL.axvspan(j - 0.48, j + 0.48, alpha=0.10, color='#c0392b')
        axL.set_xticks(xs)
        axL.set_xticklabels(orig_toks, rotation=80, ha='right', fontsize=4.5)
        axL.set_ylim(0, 1.35)
        axL.set_title(
            'Multi-Level Residual Pool Attention (Original Code)\n'
            'Level 1 = shallow syntax  |  Level 4 = deep semantics  |  '
            'Red shading = security-relevant token columns',
            fontsize=8.5, fontweight='bold'
        )
        axL.legend(fontsize=7, ncol=n_lev, loc='upper right')
        axL.grid(axis='y', alpha=0.25, linewidth=0.4)

    fname = f"{peft_name}_adv_s{sample_idx}_{case}.png"
    plt.savefig(os.path.join(odir, fname), dpi=110, bbox_inches='tight')
    plt.close()
    return fname, case


def adv_qualitative_analysis(model, adf, tok, peft_name, odir, il):
    global REFERENCE_IDX

    ods = DS(adf['original_code'].tolist(),   adf['vuln_label'].tolist(), tok)
    ads = DS(adf['adversarial_code'].tolist(), adf['vuln_label'].tolist(), tok)

    model.eval()
    fooled, robust = [], []

    for i in range(len(ods)):
        if len(fooled) >= VIZ_SAMPLES and len(robust) >= VIZ_SAMPLES:
            break
        b_o = ods[i]
        b_a = ads[i]
        lbl = b_o['label'].item()
        ids_o = b_o['ids'].unsqueeze(0).to(DEVICE)
        msk_o = b_o['mask'].unsqueeze(0).to(DEVICE)
        ids_a = b_a['ids'].unsqueeze(0).to(DEVICE)
        msk_a = b_a['mask'].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            with autocast(device_type='cuda'):
                po = unwrap(model)(ids_o, msk_o).argmax(-1).item()
                pa = unwrap(model)(ids_a, msk_a).argmax(-1).item()

        if po == lbl and pa != lbl and len(fooled) < VIZ_SAMPLES:
            fooled.append((i, lbl, po, pa))
        elif po == lbl and pa == lbl and len(robust) < VIZ_SAMPLES:
            robust.append((i, lbl, po, pa))

    if REFERENCE_IDX is None and fooled:
        REFERENCE_IDX = fooled[0][0]

    peft_store = {}

    for case_list in [fooled, robust]:
        for (idx, true_lbl, po, pa) in case_list:
            b_o       = ods[idx]
            b_a       = ads[idx]
            orig_toks = get_toks(tok, b_o['ids'], b_o['mask'])
            adv_toks  = get_toks(tok, b_a['ids'], b_a['mask'])
            orig_info = extract_attention(model, b_o['ids'], b_o['mask'], peft_name)
            adv_info  = extract_attention(model, b_a['ids'], b_a['mask'], peft_name)

            plot_adv_token_study(orig_toks, adv_toks, orig_info, adv_info,
                                 po, pa, true_lbl, peft_name, idx, odir)

            if idx == REFERENCE_IDX:
                peft_store = {
                    'orig_toks': orig_toks, 'adv_toks': adv_toks,
                    'orig_info': orig_info,  'adv_info': adv_info,
                    'pred_o': po, 'pred_a': pa, 'true_lbl': true_lbl
                }

    if not peft_store and REFERENCE_IDX is not None:
        b_o       = ods[REFERENCE_IDX]
        b_a       = ads[REFERENCE_IDX]
        orig_toks = get_toks(tok, b_o['ids'], b_o['mask'])
        adv_toks  = get_toks(tok, b_a['ids'], b_a['mask'])
        orig_info = extract_attention(model, b_o['ids'], b_o['mask'], peft_name)
        adv_info  = extract_attention(model, b_a['ids'], b_a['mask'], peft_name)
        ids_o = b_o['ids'].unsqueeze(0).to(DEVICE)
        msk_o = b_o['mask'].unsqueeze(0).to(DEVICE)
        ids_a = b_a['ids'].unsqueeze(0).to(DEVICE)
        msk_a = b_a['mask'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            with autocast(device_type='cuda'):
                po = unwrap(model)(ids_o, msk_o).argmax(-1).item()
                pa = unwrap(model)(ids_a, msk_a).argmax(-1).item()
        peft_store = {
            'orig_toks': orig_toks, 'adv_toks': adv_toks,
            'orig_info': orig_info,  'adv_info': adv_info,
            'pred_o': po, 'pred_a': pa, 'true_lbl': b_o['label'].item()
        }

    ADV_ATTN_STORE[peft_name] = peft_store


def plot_cross_peft_comparison(store, odir):
    available = [p for p in PEFTS if p in store and store[p].get('orig_toks')]
    if not available:
        return

    n  = len(available)
    fig, axes = plt.subplots(n, 2, figsize=(24, n * 3.2))
    if n == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(
        'Cross-PEFT Adversarial Pool Attention Comparison\n'
        'Same reference adversarial sample evaluated under all PEFT variants\n'
        'Red bars = security-relevant tokens  |  '
        'FOOLED = attack succeeds  |  ROBUST = model holds',
        fontsize=12, fontweight='bold', y=1.02
    )

    cc = {'FOOLED': '#c0392b', 'ROBUST': '#27ae60', 'WRONG': '#e67e22'}

    for ri, peft_name in enumerate(available):
        d         = store[peft_name]
        orig_toks = d.get('orig_toks', [])
        adv_toks  = d.get('adv_toks',  [])
        o_info    = d.get('orig_info', {})
        a_info    = d.get('adv_info',  {})
        po        = d.get('pred_o', -1)
        pa        = d.get('pred_a', -1)
        tl        = d.get('true_lbl', -1)

        o_pool = o_info.get('agg_pool', np.zeros(len(orig_toks)))
        a_pool = a_info.get('agg_pool', np.zeros(len(adv_toks)))

        case = ('FOOLED' if po == tl and pa != tl else
                'ROBUST' if po == tl and pa == tl else 'WRONG')

        for ci, (toks, pool_w, suffix, pred) in enumerate([
            (orig_toks, o_pool, 'ORIGINAL',    po),
            (adv_toks,  a_pool, 'ADVERSARIAL', pa)
        ]):
            ax     = axes[ri, ci]
            nt     = len(toks)
            mx     = pool_w[:nt].max() + 1e-9
            colors = ['#c0392b' if is_sec_token(t) else '#95a5a6' for t in toks]
            ax.bar(np.arange(nt), pool_w[:nt] / mx, color=colors,
                   alpha=0.85, edgecolor='black', linewidth=0.15)
            ax.set_xticks(np.arange(nt))
            ax.set_xticklabels(toks, rotation=80, ha='right', fontsize=3.8)
            ax.set_title(
                f"{peft_name}  ·  {suffix}  "
                f"[pred: {'Vuln' if pred == 1 else 'Clean'}]  "
                f"{'[' + case + ']' if ci == 1 else ''}",
                fontsize=7.5, fontweight='bold',
                color=cc.get(case, 'black') if ci == 1 else 'black'
            )
            ax.set_ylim(0, 1.3)
            ax.grid(axis='y', alpha=0.2, linewidth=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(odir, 'cross_peft_adv_comparison.png'),
                dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  Cross-PEFT comparison saved -> cross_peft_adv_comparison.png")


def qualitative(model, adf, tok, name, odir, il):
    ods = DS(adf['original_code'].tolist(),   adf['vuln_label'].tolist(), tok)
    ads = DS(adf['adversarial_code'].tolist(), adf['vuln_label'].tolist(), tok)

    ri = wi = None
    for i in range(len(ods)):
        b_o  = ods[i]
        ids  = b_o['ids'].unsqueeze(0).to(DEVICE)
        mask = b_o['mask'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            with autocast(device_type='cuda'):
                pred = unwrap(model)(ids, mask).argmax(-1).item()
        lab = b_o['label'].item()
        if ri is None and pred == lab:  ri = i
        if wi is None and pred != lab:  wi = i
        if ri is not None and wi is not None:
            break

    for idx, case in [(ri, 'CORRECT'), (wi, 'WRONG')]:
        if idx is None:
            continue
        b_o      = ods[idx]
        b_a      = ads[idx]
        toks     = get_toks(tok, b_o['ids'], b_o['mask'])
        adv_toks = get_toks(tok, b_a['ids'], b_a['mask'])
        try:
            info     = extract_attention(model, b_o['ids'], b_o['mask'], name)
            adv_info = extract_attention(model, b_a['ids'], b_a['mask'], name)
            if il and 'level_pool' in info:
                n_lev = len(info['level_pool'])
                n_t   = len(toks)
                fig, axes = plt.subplots(n_lev, 1, figsize=(max(10, n_t // 3), n_lev * 2 + 1))
                if n_lev == 1:
                    axes = [axes]
                for li, (ax, lw) in enumerate(zip(axes, info['level_pool'])):
                    nw = lw[:n_t] / (lw[:n_t].max() + 1e-9)
                    ax.bar(np.arange(n_t), nw, color=_sec_colors(toks), alpha=0.85,
                           edgecolor='k', linewidth=0.2)
                    ax.set_xticks(np.arange(n_t))
                    ax.set_xticklabels(toks, rotation=80, ha='right', fontsize=5)
                    ax.set_title(f'LAPPEFT Level {li+1} Pool Attention | ADV Sample #{idx} [{case}]',
                                 fontsize=8)
                    ax.set_ylim(0, 1.25)
                fig.suptitle(f"LAPPEFT {case} | Adversarial Dataset Sample #{idx}",
                             fontweight='bold', fontsize=9)
                plt.tight_layout()
                plt.savefig(os.path.join(odir, f"LAPPEFT_{case}_pool.png"),
                            dpi=100, bbox_inches='tight')
                plt.close()
            else:
                bb_attn = info.get('bb_attn', np.zeros((2, 2)))
                adv_bb  = adv_info.get('bb_attn', np.zeros((2, 2)))
                n       = len(toks)
                na      = len(adv_toks)
                fig, axes2 = plt.subplots(1, 2, figsize=(max(16, (n + na) // 3), max(7, n // 3)))
                ax0, ax1 = axes2
                im0 = ax0.imshow(bb_attn[:n, :n], cmap='YlOrRd', aspect='auto')
                plt.colorbar(im0, ax=ax0, shrink=0.7)
                ax0.set_xticks(range(n)); ax0.set_yticks(range(n))
                ax0.set_xticklabels(toks, rotation=90, fontsize=5)
                ax0.set_yticklabels(toks, fontsize=5)
                ax0.set_title(f"{name} {case} ORIGINAL | ADV Sample #{idx}",
                              fontsize=9, fontweight='bold')
                im1 = ax1.imshow(adv_bb[:na, :na], cmap='YlOrRd', aspect='auto')
                plt.colorbar(im1, ax=ax1, shrink=0.7)
                ax1.set_xticks(range(na)); ax1.set_yticks(range(na))
                ax1.set_xticklabels(adv_toks, rotation=90, fontsize=5)
                ax1.set_yticklabels(adv_toks, fontsize=5)
                ax1.set_title(f"{name} {case} ADVERSARIAL | ADV Sample #{idx}",
                              fontsize=9, fontweight='bold')
                plt.tight_layout()
                plt.savefig(os.path.join(odir, f"{name}_{case}_attn.png"),
                            dpi=100, bbox_inches='tight')
                plt.close()
        except Exception as e:
            print(f"  [viz err | {name} {case}]: {e}")


def adv_eval(model, adf, tok):
    ods = DS(adf['original_code'].tolist(),   adf['vuln_label'].tolist(), tok)
    ads = DS(adf['adversarial_code'].tolist(), adf['vuln_label'].tolist(), tok)
    ol  = DataLoader(ods, batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    al  = DataLoader(ads, batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
    yl, ypo, pro = infer(model, ol)
    _,  ypa, pra = infer(model, al)
    om  = mets(yl, ypo, pro)
    am  = mets(yl, ypa, pra)
    cor = (ypo == yl)
    asr = float(((cor) & (ypo != ypa)).sum() / (cor.sum() + 1e-9))
    return om, am, asr, float((ypa == yl).mean()), int((ypo != ypa).sum())


def plot_summary(res, odir):
    names = list(res.keys())
    mk    = ['Acc', 'Pre', 'Rec', 'F1', 'AUC']
    x     = np.arange(len(names))
    w     = 0.15
    cs    = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6', '#e67e22']

    fig, axes = plt.subplots(1, 2, figsize=(20, 6))
    for ax, mode, ttl in zip(axes, ['om', 'am'], ['Original', 'Adversarial']):
        for j, (k, c) in enumerate(zip(mk, cs)):
            ax.bar(x + (j - 2) * w, [res[n][mode][k] for n in names], w,
                   color=c, alpha=0.85, label=k, edgecolor='k', linewidth=0.3)
        ax.set_xticks(x); ax.set_xticklabels(names, rotation=30, ha='right')
        ax.set_ylim(0, 1.15)
        ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
        ax.set_title(f"{ttl} Code Metrics", fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(odir, 'metrics.png'), dpi=120, bbox_inches='tight')
    plt.close()

    data = np.array([[res[n]['om'][k] - res[n]['am'][k] for k in mk] for n in names])
    fig, ax = plt.subplots(figsize=(9, max(4, len(names) * 0.6 + 1)))
    im = ax.imshow(data, cmap='RdYlGn_r', aspect='auto', vmin=-0.3, vmax=0.3)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(mk))); ax.set_yticks(range(len(names)))
    ax.set_xticklabels(mk); ax.set_yticklabels(names)
    for i in range(len(names)):
        for j in range(len(mk)):
            ax.text(j, i, f'{data[i, j]:+.3f}', ha='center', va='center', fontsize=8)
    ax.set_title('Metric Drop (Orig - Adv) per PEFT', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(odir, 'delta.png'), dpi=120, bbox_inches='tight')
    plt.close()

    asrs  = [res[n]['asr']       for n in names]
    robs  = [res[n]['rob']       for n in names]
    oaccs = [res[n]['om']['Acc'] for n in names]
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.bar(x - w, oaccs, w, color='#2ecc71', label='Orig Acc',   edgecolor='k', linewidth=0.3)
    ax.bar(x,     robs,  w, color='#3498db', label='Robust Acc', edgecolor='k', linewidth=0.3)
    ax.bar(x + w, asrs,  w, color='#e74c3c', label='ASR',        edgecolor='k', linewidth=0.3)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=25, ha='right')
    ax.set_ylim(0, 1.15)
    ax.legend(); ax.grid(axis='y', alpha=0.3)
    ax.set_title('Adversarial Robustness Summary', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(odir, 'robustness.png'), dpi=120, bbox_inches='tight')
    plt.close()


def plot_radar(res, odir):
    names  = list(res.keys())
    mk     = ['Acc', 'Pre', 'Rec', 'F1', 'AUC']
    n      = len(mk)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, axes = plt.subplots(1, 2, figsize=(16, 7), subplot_kw=dict(polar=True))
    fig.suptitle('PEFT Comparison Radar', fontweight='bold', fontsize=13)
    pal = plt.cm.tab10(np.linspace(0, 1, len(names)))
    for ax, mode, ttl in zip(axes, ['om', 'am'], ['Original', 'Adversarial']):
        ax.set_title(ttl, fontweight='bold', pad=20)
        for mi, nm in enumerate(names):
            vals = [res[nm][mode][k] for k in mk] + [res[nm][mode][mk[0]]]
            ax.plot(angles, vals, color=pal[mi], linewidth=2, label=nm)
            ax.fill(angles, vals, color=pal[mi], alpha=0.07)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(mk, size=9)
        ax.set_ylim(0, 1)
        ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(odir, 'radar.png'), dpi=120, bbox_inches='tight')
    plt.close()


def main():
    global REFERENCE_IDX, ADV_ATTN_STORE
    REFERENCE_IDX  = None
    ADV_ATTN_STORE = {}
    os.makedirs(OUT, exist_ok=True)

    tok = AutoTokenizer.from_pretrained(BB, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token    = tok.eos_token
        tok.pad_token_id = tok.eos_token_id

    tr  = pd.read_csv(TRAIN_CSV).rename(columns={'func': 'code'})
    te  = pd.read_csv(TEST_CSV).rename(columns={'func': 'code'})
    adf = (pd.read_csv(ADV_CSV)
           .dropna(subset=['original_code', 'adversarial_code', 'vuln_label'])
           .reset_index(drop=True))
    print(f"ADV rows: {len(adf)} | cols: {list(adf.columns)}")

    tr_df, val_df = train_test_split(tr, test_size=0.2, stratify=tr['label'],
                                     random_state=SD)
    _,     te_df  = train_test_split(te, test_size=0.5, stratify=te['label'],
                                     random_state=SD)
    print(f"Train: {len(tr_df)}  Val: {len(val_df)}  Test: {len(te_df)}  "
          f"Device: {DEVICE}  N_GPU: {N_GPU}")

    def mkds(df):
        return DS(df['code'].tolist(), df['label'].tolist(), tok)

    tr_ds  = mkds(tr_df)
    val_ds = mkds(val_df)
    te_ds  = mkds(te_df)

    tr_dl  = DataLoader(tr_ds,  batch_size=BS, shuffle=True,  num_workers=NUM_WORKERS,
                        pin_memory=True, drop_last=True)
    val_dl = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=True)
    te_dl  = DataLoader(te_ds,  batch_size=BS, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=True)

    vc   = tr_df['label'].value_counts().sort_index().values
    cwr  = torch.tensor([1 / c for c in vc], dtype=torch.float).to(DEVICE)
    cw   = cwr / cwr.sum() * 2
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.05)

    completed = load_run_checkpoint()
    all_res   = {}

    for peft_name, r in completed.items():
        all_res[peft_name] = r
        print(f"  [SKIP — already done from checkpoint] PEFT: {peft_name}")

    for peft in PEFTS:
        if peft in completed:
            continue

        print(f"\n{'=' * 58}\n  PEFT: {peft}\n{'=' * 58}")
        il    = (peft == 'LAPPEFT')
        model = (LAPMod() if il else PEFTMod(peft))

        if N_GPU > 1:
            model = nn.DataParallel(model, device_ids=list(range(N_GPU)))
        model = model.to(DEVICE)

        tp = [p for p in model.parameters() if p.requires_grad]
        print(f"  Trainable params: {sum(p.numel() for p in tp):,}   "
              f"GPU replicas: {N_GPU}")

        opt    = torch.optim.AdamW(tp, lr=LR, weight_decay=0.05)
        steps  = (len(tr_dl) // GA) * EP
        ws     = int(WR * steps)
        sched  = get_cosine_schedule_with_warmup(opt, ws, steps)
        scaler = GradScaler(device='cuda')

        bf  = -1.0
        bst = None
        ni  = 0

        for ep in range(1, EP + 1):
            tl, tm = train_ep(model, tr_dl, crit, opt, sched, cw, il, scaler)
            vl, vm = eval_ep(model, val_dl, crit)
            note   = ' <--' if vm['F1'] > bf else ''
            if vm['F1'] > bf:
                bf  = vm['F1']
                bst = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                ni  = 0
            else:
                ni += 1
            print(f"  Ep{ep:2d}  TrL:{tl:.4f} F1:{tm['F1']:.4f}  "
                  f"VaL:{vl:.4f} F1:{vm['F1']:.4f} AUC:{vm['AUC']:.4f}{note}")
            if ni >= PT:
                print("  Early stop")
                break

        model.load_state_dict({k: v.to(DEVICE) for k, v in bst.items()})
        _, te_m = eval_ep(model, te_dl, crit)
        print(f"  Test: {' '.join(f'{k}={v:.4f}' for k, v in te_m.items())}")

        om, am, asr, rob, flip = adv_eval(model, adf, tok)
        print(f"  Adv:  ASR={asr:.4f}  RobAcc={rob:.4f}  Flipped={flip}")

        pdir = os.path.join(OUT, peft)
        os.makedirs(pdir, exist_ok=True)

        qualitative(model, adf, tok, peft, pdir, il)
        adv_qualitative_analysis(model, adf, tok, peft, pdir, il)

        all_res[peft] = {'om': om, 'am': am, 'te': te_m,
                         'asr': asr, 'rob': rob, 'flip': flip}

        torch.save({'s': bst, 'peft': peft}, os.path.join(pdir, 'model.pt'))

        save_run_checkpoint(all_res)
        print(f"  Checkpoint saved after {peft}")

        del model, opt, sched, scaler, bst
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()

    plot_cross_peft_comparison(ADV_ATTN_STORE, OUT)
    plot_summary(all_res, OUT)
    plot_radar(all_res, OUT)

    rows = []
    for nm, r in all_res.items():
        row = {'PEFT': nm, 'ASR': round(r['asr'], 4),
               'RobAcc': round(r['rob'], 4), 'Flipped': r['flip']}
        for k in ['Acc', 'Pre', 'Rec', 'F1', 'AUC']:
            row[f'Orig_{k}'] = round(r['om'][k], 4)
            row[f'Adv_{k}']  = round(r['am'][k], 4)
        rows.append(row)

    pd.DataFrame(rows).to_csv(os.path.join(OUT, 'summary.csv'), index=False)
    print(f"\nDone  ->  {OUT}")


if __name__ == '__main__':
    main()

# Hieradapter Inference on adversarial_vuln dataset analysis for code vulnerability (UniXcoder)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
MODEL_NAME  = "microsoft/unixcoder-base"
MODEL_PATH  = "ults/best_proposed_model.pt"
ADV_CSV     = "/adversarial_vuln_WITH_LABELS.csv"
OUT_DIR     = "/analysis_results"
MAX_LENGTH  = 512
D_MODEL     = 768
BATCH_SIZE  = 16
BOTTLENECK  = 6
NUM_LEVELS  = 4
LAYERS_LEVEL= 3
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
class RawCodeDataset(Dataset):
    def __init__(self, codes, labels, tokenizer):
        self.codes     = [str(c) for c in codes]
        self.labels    = list(labels)
        self.tokenizer = tokenizer
    def __len__(self):
        return len(self.codes)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }
class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)
    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)
class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)
    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted
class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)
    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))
class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        for p in self.encoder.parameters():
            p.requires_grad = False
        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])
        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK) for _ in range(NUM_LEVELS)
        ])
        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])
        self.fusion      = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)
        self.classifier  = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )
    def encode(self, input_ids, attention_mask):
        out        = self.encoder(input_ids=input_ids, attention_mask=attention_mask, output_hidden_states=True)
        hidden     = out.hidden_states
        num_layers = len(hidden)
        base       = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)
        level_indices = [min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)]
        level_cls     = [self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask) for lvl in range(NUM_LEVELS)]
        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))
        return self.fusion(base, residuals)
    def forward(self, input_ids, attention_mask, cls_embed=None):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        return self.classifier(self.encode(input_ids, attention_mask))
def compute_clf_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }
def run_inference(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            logits         = model(input_ids, attention_mask)
            probs          = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds          = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)
def compute_adversarial_metrics(orig_labels, orig_preds, orig_probs, adv_preds, adv_probs, adv_df):
    orig_labels  = np.array(orig_labels)
    orig_preds   = np.array(orig_preds)
    orig_probs   = np.array(orig_probs)
    adv_preds    = np.array(adv_preds)
    adv_probs    = np.array(adv_probs)
    n            = len(orig_labels)
    orig_correct = (orig_preds == orig_labels)
    adv_correct  = (adv_preds  == orig_labels)
    flipped      = (orig_preds != adv_preds)
    success_mask = orig_correct & flipped
    asr          = float(success_mask.sum() / (orig_correct.sum() + 1e-9))
    flip_rate    = float(flipped.sum() / n)
    robust_acc   = float(adv_correct.sum() / n)
    orig_acc     = float(orig_correct.sum() / n)
    acc_drop     = orig_acc - robust_acc
    conf_drop_all  = float((orig_probs - adv_probs).mean())
    conf_drop_succ = float((orig_probs[success_mask] - adv_probs[success_mask]).mean()) if success_mask.any() else 0.0
    orig_conf_correct = float(orig_probs[orig_correct].mean()) if orig_correct.any() else 0.0
    adv_conf_correct  = float(adv_probs[adv_correct].mean())  if adv_correct.any()  else 0.0
    vuln_mask    = orig_labels == 1
    nonvuln_mask = orig_labels == 0
    asr_vuln     = float((success_mask & vuln_mask).sum()    / ((orig_correct & vuln_mask).sum()    + 1e-9))
    asr_nonvuln  = float((success_mask & nonvuln_mask).sum() / ((orig_correct & nonvuln_mask).sum() + 1e-9))
    flip_to_vuln    = int(((orig_preds == 0) & (adv_preds == 1)).sum())
    flip_to_nonvuln = int(((orig_preds == 1) & (adv_preds == 0)).sum())
    def _slice_stats(mask_series):
        out = {}
        for key in mask_series['adv_df'].columns if False else []:
            pass
        return out
    def _group_stats(col):
        result = {}
        if col not in adv_df.columns:
            return result
        for val in adv_df[col].dropna().unique():
            idx = (adv_df[col] == val).values
            if idx.sum() == 0:
                continue
            oc = orig_correct[idx]
            sm = success_mask[idx]
            result[str(val)] = {
                'n':            int(idx.sum()),
                'asr':          float(sm.sum() / (oc.sum() + 1e-9)),
                'flip_rate':    float(flipped[idx].sum() / idx.sum()),
                'robust_acc':   float((adv_preds[idx] == orig_labels[idx]).mean()),
                'orig_acc':     float(oc.mean()),
                'conf_drop':    float((orig_probs[idx] - adv_probs[idx]).mean()),
                'n_flipped':    int(flipped[idx].sum()),
                'n_successful': int(sm.sum()),
            }
        return result
    def _ast_stats():
        result = {}
        col = 'attack_success_type'
        if col not in adv_df.columns:
            return result
        for val in adv_df[col].dropna().unique():
            idx = (adv_df[col] == val).values
            if idx.sum() == 0:
                continue
            sm = success_mask[idx]
            result[str(val)] = {
                'n':            int(idx.sum()),
                'n_success':    int(sm.sum()),
                'success_rate': float(sm.sum() / idx.sum()),
                'flip_rate':    float(flipped[idx].sum() / idx.sum()),
                'conf_drop':    float((orig_probs[idx] - adv_probs[idx]).mean()),
            }
        return result
    return {
        'n_total':               n,
        'n_orig_correct':        int(orig_correct.sum()),
        'n_adv_correct':         int(adv_correct.sum()),
        'n_flipped':             int(flipped.sum()),
        'n_successful_attacks':  int(success_mask.sum()),
        'attack_success_rate':   asr,
        'flip_rate':             flip_rate,
        'robust_accuracy':       robust_acc,
        'original_accuracy':     orig_acc,
        'accuracy_drop':         acc_drop,
        'mean_conf_drop':        conf_drop_all,
        'mean_conf_drop_successful': conf_drop_succ,
        'orig_conf_on_correct':  orig_conf_correct,
        'adv_conf_on_correct':   adv_conf_correct,
        'asr_on_vulnerable':     asr_vuln,
        'asr_on_nonvulnerable':  asr_nonvuln,
        'label_flip_to_vuln':    flip_to_vuln,
        'label_flip_to_nonvuln': flip_to_nonvuln,
        'per_attack_method':     _group_stats('attack_method'),
        'per_category':          _group_stats('category'),
        'per_cwe':               _group_stats('cwe'),
        'per_attack_success_type': _ast_stats(),
    }
def _bar_vals(ax, bars, fmt='{:.4f}', fontsize=9):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + ax.get_ylim()[1] * 0.01,
                fmt.format(bar.get_height()),
                ha='center', va='bottom', fontsize=fontsize, fontweight='bold')
def plot_adv_overview(m, save_path):
    rate_labels = ['Orig\nAccuracy', 'Robust\nAccuracy', 'Attack\nSuccess\nRate', 'Flip\nRate', 'Acc\nDrop']
    rate_vals   = [m['original_accuracy'], m['robust_accuracy'], m['attack_success_rate'],
                   m['flip_rate'], m['accuracy_drop']]
    rate_colors = ['#2ecc71', '#3498db', '#e74c3c', '#e67e22', '#9b59b6']
    cnt_labels  = ['Orig\nCorrect', 'Robust\nCorrect', 'Flipped', 'Successful\nAttacks',
                   'Flip→\nVuln', 'Flip→\nNon-Vuln']
    cnt_vals    = [m['n_orig_correct'], m['n_adv_correct'], m['n_flipped'],
                   m['n_successful_attacks'], m['label_flip_to_vuln'], m['label_flip_to_nonvuln']]
    cnt_colors  = ['#27ae60', '#2980b9', '#e67e22', '#c0392b', '#8e44ad', '#16a085']
    fig, axes   = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('LAPPEFT — Adversarial Robustness Overview', fontsize=14, fontweight='bold')
    bars = axes[0].bar(rate_labels, rate_vals, color=rate_colors, alpha=0.88, edgecolor='black', linewidth=0.6)
    axes[0].set_ylim(0, 1.18)
    axes[0].set_ylabel('Rate / Score', fontsize=11)
    axes[0].set_title('Core Adversarial Rates', fontweight='bold', fontsize=11)
    axes[0].grid(axis='y', alpha=0.3)
    _bar_vals(axes[0], bars)
    bars2 = axes[1].bar(cnt_labels, cnt_vals, color=cnt_colors, alpha=0.88, edgecolor='black', linewidth=0.6)
    axes[1].set_ylabel('Sample Count', fontsize=11)
    axes[1].set_title('Sample Counts', fontweight='bold', fontsize=11)
    axes[1].grid(axis='y', alpha=0.3)
    _bar_vals(axes[1], bars2, fmt='{:.0f}')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_orig_vs_adv_cm(orig_labels, orig_preds, adv_preds, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(11, 5))
    fig.suptitle('Original vs Adversarial Confusion Matrices', fontsize=13, fontweight='bold')
    for ax, preds, title in zip(axes, [orig_preds, adv_preds], ['Original Code', 'Adversarial Code']):
        cm     = confusion_matrix(orig_labels, preds, labels=[0, 1])
        im     = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        plt.colorbar(im, ax=ax)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Pred 0', 'Pred 1'])
        ax.set_yticklabels(['True 0', 'True 1'])
        thresh = cm.max() / 2.0
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                        color='white' if cm[i, j] > thresh else 'black',
                        fontsize=14, fontweight='bold')
        ax.set_title(title, fontweight='bold', fontsize=11)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_radar_orig_vs_adv(orig_m, adv_m, save_path):
    keys   = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    ax.set_title('Original vs Adversarial Radar', fontweight='bold', pad=25, fontsize=13)
    for m, label, color in [(orig_m, 'Original Code', '#2ecc71'), (adv_m, 'Adversarial Code', '#e74c3c')]:
        vals  = [m[k] for k in keys] + [m[keys[0]]]
        ax.plot(angles, vals, color=color, linewidth=2.5, label=label)
        ax.fill(angles, vals, color=color, alpha=0.12)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=10)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_metric_drop_bar(orig_m, adv_m, save_path):
    keys   = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC',
              'Precision', 'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity']
    orig_v = [orig_m[k] for k in keys]
    adv_v  = [adv_m[k]  for k in keys]
    drop_v = [o - a for o, a in zip(orig_v, adv_v)]
    x      = np.arange(len(keys))
    width  = 0.28
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    fig.suptitle('Classification Metrics: Original vs Adversarial Code', fontsize=14, fontweight='bold')
    axes[0].bar(x - width / 2, orig_v, width=width, color='#2ecc71', alpha=0.88,
                edgecolor='black', linewidth=0.5, label='Original Code')
    axes[0].bar(x + width / 2, adv_v,  width=width, color='#e74c3c', alpha=0.88,
                edgecolor='black', linewidth=0.5, label='Adversarial Code')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(keys, rotation=35, ha='right', fontsize=9)
    axes[0].set_ylim(0, 1.15)
    axes[0].set_ylabel('Score', fontsize=11)
    axes[0].set_title('Side-by-side Metric Comparison', fontweight='bold', fontsize=11)
    axes[0].legend(fontsize=10)
    axes[0].grid(axis='y', alpha=0.3)
    bar_colors = ['#e74c3c' if d >= 0 else '#2ecc71' for d in drop_v]
    axes[1].bar(x, drop_v, color=bar_colors, alpha=0.88, edgecolor='black', linewidth=0.5)
    axes[1].axhline(0, color='black', linewidth=1.2)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(keys, rotation=35, ha='right', fontsize=9)
    axes[1].set_ylabel('Drop (Orig − Adv)', fontsize=11)
    axes[1].set_title('Metric Drop Under Attack', fontweight='bold', fontsize=11)
    axes[1].grid(axis='y', alpha=0.3)
    for bx, d in zip(x, drop_v):
        axes[1].text(bx, d + (0.005 if d >= 0 else -0.015),
                     f'{d:+.3f}', ha='center', va='bottom', fontsize=7.5, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_conf_shift(orig_probs, adv_probs, orig_labels, save_path):
    conf_drop    = orig_probs - adv_probs
    vuln_drop    = conf_drop[orig_labels == 1]
    nonvuln_drop = conf_drop[orig_labels == 0]
    fig, axes    = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Confidence Shift Under Adversarial Attack', fontsize=13, fontweight='bold')
    axes[0].hist(conf_drop, bins=50, color='#3498db', alpha=0.82, edgecolor='black', linewidth=0.3)
    axes[0].axvline(0, color='red', linestyle='--', linewidth=1.5, label='No change')
    axes[0].axvline(conf_drop.mean(), color='orange', linestyle='--', linewidth=1.5,
                    label=f'Mean={conf_drop.mean():.3f}')
    axes[0].set_xlabel('Confidence Drop (orig − adv)', fontsize=10)
    axes[0].set_ylabel('Count', fontsize=10)
    axes[0].set_title('Overall Confidence Drop', fontweight='bold', fontsize=10)
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.3)
    axes[1].hist(vuln_drop,    bins=40, color='#e74c3c', alpha=0.7, edgecolor='black',
                 linewidth=0.3, label=f'Vulnerable (n={len(vuln_drop)})')
    axes[1].hist(nonvuln_drop, bins=40, color='#2ecc71', alpha=0.7, edgecolor='black',
                 linewidth=0.3, label=f'Non-Vulnerable (n={len(nonvuln_drop)})')
    axes[1].axvline(0, color='black', linestyle='--', linewidth=1.2)
    axes[1].set_xlabel('Confidence Drop', fontsize=10)
    axes[1].set_ylabel('Count', fontsize=10)
    axes[1].set_title('Conf Drop by True Label', fontweight='bold', fontsize=10)
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.3)
    axes[2].scatter(orig_probs, adv_probs, alpha=0.4, s=8, c=orig_labels, cmap='RdYlGn', edgecolors='none')
    axes[2].plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='No change line')
    axes[2].set_xlabel('Original Confidence (Prob Vuln)', fontsize=10)
    axes[2].set_ylabel('Adversarial Confidence (Prob Vuln)', fontsize=10)
    axes[2].set_title('Orig vs Adv Confidence Scatter', fontweight='bold', fontsize=10)
    axes[2].legend(fontsize=8)
    axes[2].grid(alpha=0.3)
    axes[2].set_xlim(-0.02, 1.02)
    axes[2].set_ylim(-0.02, 1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_per_group(group_dict, group_col_name, title_suffix, save_path):
    if not group_dict:
        return
    groups      = list(group_dict.keys())
    metric_keys = ['asr', 'flip_rate', 'robust_acc', 'orig_acc', 'conf_drop']
    metric_lbls = ['Attack Success Rate', 'Flip Rate', 'Robust Accuracy', 'Orig Accuracy', 'Conf Drop']
    colors      = plt.cm.Set2(np.linspace(0, 1, len(metric_keys)))
    x           = np.arange(len(groups))
    width       = 0.15
    fig, ax     = plt.subplots(figsize=(max(14, len(groups) * 2), 7))
    for i, (mk, ml) in enumerate(zip(metric_keys, metric_lbls)):
        vals    = [group_dict[g][mk] for g in groups]
        offsets = x + (i - len(metric_keys) / 2) * width + width / 2
        ax.bar(offsets, vals, width=width * 0.9, color=colors[i],
               alpha=0.88, edgecolor='black', linewidth=0.4, label=ml)
    ax.set_xticks(x)
    ax.set_xticklabels(groups, rotation=35, ha='right', fontsize=9)
    ax.set_ylim(0, 1.18)
    ax.set_ylabel('Score / Rate', fontsize=11)
    ax.set_title(f'LAPPEFT — Adversarial Metrics per {title_suffix}', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=9, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_attack_success_type(per_ast, save_path):
    if not per_ast:
        return
    types   = list(per_ast.keys())
    n_vals  = [per_ast[t]['n']            for t in types]
    sr_vals = [per_ast[t]['success_rate'] for t in types]
    fr_vals = [per_ast[t]['flip_rate']    for t in types]
    cd_vals = [per_ast[t]['conf_drop']    for t in types]
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.suptitle('Analysis by Attack Success Type', fontsize=14, fontweight='bold')
    axes[0].pie(n_vals, labels=types, colors=plt.cm.Set3(np.linspace(0, 1, len(types))),
                autopct='%1.1f%%', startangle=140, textprops={'fontsize': 8})
    axes[0].set_title('Sample Distribution by Attack Success Type', fontweight='bold', fontsize=10)
    x     = np.arange(len(types))
    width = 0.25
    axes[1].bar(x - width, sr_vals, width=width, color='#e74c3c', alpha=0.88,
                edgecolor='black', linewidth=0.4, label='Success Rate')
    axes[1].bar(x,         fr_vals, width=width, color='#e67e22', alpha=0.88,
                edgecolor='black', linewidth=0.4, label='Flip Rate')
    axes[1].bar(x + width, cd_vals, width=width, color='#9b59b6', alpha=0.88,
                edgecolor='black', linewidth=0.4, label='Conf Drop')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(types, rotation=30, ha='right', fontsize=8)
    axes[1].set_ylim(0, 1.18)
    axes[1].set_ylabel('Rate / Score', fontsize=11)
    axes[1].set_title('Rates by Attack Success Type', fontweight='bold', fontsize=10)
    axes[1].legend(fontsize=9)
    axes[1].grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_heatmap_method_cwe(per_attack, per_cwe, save_path):
    if not per_attack or not per_cwe:
        return
    mk      = ['asr', 'flip_rate', 'robust_acc', 'orig_acc', 'conf_drop']
    ml      = ['ASR', 'Flip Rate', 'Robust Acc', 'Orig Acc', 'Conf Drop']
    methods = list(per_attack.keys())
    cwes    = list(per_cwe.keys())
    data_m  = np.array([[per_attack[m][k] for k in mk] for m in methods])
    data_c  = np.array([[per_cwe[c][k]    for k in mk] for c in cwes])
    fig, axes = plt.subplots(1, 2, figsize=(20, max(5, max(len(methods), len(cwes)) * 0.55 + 2)))
    fig.suptitle('LAPPEFT — Adversarial Heatmaps: Attack Method & CWE', fontsize=13, fontweight='bold')
    for ax, data, ylabels, subtitle in [
        (axes[0], data_m, methods, 'Per Attack Method'),
        (axes[1], data_c, cwes,    'Per CWE')
    ]:
        im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
        plt.colorbar(im, ax=ax, fraction=0.04, pad=0.04)
        ax.set_xticks(range(len(ml)))
        ax.set_yticks(range(len(ylabels)))
        ax.set_xticklabels(ml, rotation=30, ha='right', fontsize=9)
        ax.set_yticklabels(ylabels, fontsize=9)
        ax.set_title(subtitle, fontweight='bold', fontsize=10)
        for i in range(len(ylabels)):
            for j in range(len(mk)):
                ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center',
                        fontsize=7, fontweight='bold',
                        color='black' if 0.25 < data[i, j] < 0.75 else 'white')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_metrics_heatmap_orig_vs_adv(orig_m, adv_m, save_path):
    keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC',
            'Precision', 'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity', 'FPR', 'FNR']
    data = np.array([[orig_m[k] for k in keys], [adv_m[k] for k in keys]])
    fig, ax = plt.subplots(figsize=(18, 3.5))
    im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
    ax.set_xticks(range(len(keys)))
    ax.set_yticks([0, 1])
    ax.set_xticklabels(keys, rotation=35, ha='right', fontsize=10)
    ax.set_yticklabels(['Original Code', 'Adversarial Code'], fontsize=10)
    ax.set_title('LAPPEFT — Classification Metrics Heatmap: Original vs Adversarial', fontsize=13, fontweight='bold')
    for i, row_data in enumerate(data):
        for j, val in enumerate(row_data):
            ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8, fontweight='bold',
                    color='black' if 0.25 < val < 0.75 else 'white')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_asr_label_class(adv_m, save_path):
    labels_bar = ['ASR on\nVulnerable', 'ASR on\nNon-Vulnerable',
                  'Overall\nASR', 'Orig Acc\nVuln', 'Robust Acc\nAll']
    vals_bar   = [
        adv_m['asr_on_vulnerable'],
        adv_m['asr_on_nonvulnerable'],
        adv_m['attack_success_rate'],
        adv_m['original_accuracy'],
        adv_m['robust_accuracy'],
    ]
    colors_bar = ['#e74c3c', '#3498db', '#9b59b6', '#2ecc71', '#1abc9c']
    fig, ax    = plt.subplots(figsize=(10, 6))
    bars       = ax.bar(labels_bar, vals_bar, color=colors_bar, alpha=0.88, edgecolor='black', linewidth=0.6)
    ax.set_ylim(0, 1.18)
    ax.set_ylabel('Rate', fontsize=11)
    ax.set_title('LAPPEFT — ASR by Vulnerability Class', fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    _bar_vals(ax, bars)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_conf_asr_per_method_scatter(per_attack, save_path):
    if not per_attack:
        return
    methods    = list(per_attack.keys())
    asr_vals   = [per_attack[m]['asr']       for m in methods]
    conf_vals  = [per_attack[m]['conf_drop']  for m in methods]
    n_vals     = [per_attack[m]['n']          for m in methods]
    colors     = plt.cm.tab10(np.linspace(0, 1, len(methods)))
    fig, ax    = plt.subplots(figsize=(9, 7))
    for i, m in enumerate(methods):
        ax.scatter(asr_vals[i], conf_vals[i], s=max(n_vals[i] / max(n_vals) * 400, 60),
                   color=colors[i], edgecolors='black', linewidths=0.8, zorder=5, label=m)
        ax.annotate(m, (asr_vals[i], conf_vals[i]),
                    textcoords='offset points', xytext=(8, 4), fontsize=8)
    ax.set_xlabel('Attack Success Rate (ASR)', fontsize=11)
    ax.set_ylabel('Mean Confidence Drop', fontsize=11)
    ax.set_title('ASR vs Confidence Drop per Attack Method', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.02, 1.05)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def plot_summary_table(adv_metrics, orig_clf_m, adv_clf_m, save_path):
    scalar_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC',
                   'Precision', 'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity', 'FPR', 'FNR']
    clf_rows = [[k, f"{orig_clf_m[k]:.4f}", f"{adv_clf_m[k]:.4f}",
                 f"{orig_clf_m[k] - adv_clf_m[k]:+.4f}"] for k in scalar_keys]
    adv_rows = [
        ['Total Samples',          str(adv_metrics['n_total']),                          '', ''],
        ['Orig Correct',           str(adv_metrics['n_orig_correct']),                   '', ''],
        ['Robust Correct',         str(adv_metrics['n_adv_correct']),                    '', ''],
        ['Predictions Flipped',    str(adv_metrics['n_flipped']),                        '', ''],
        ['Successful Attacks',     str(adv_metrics['n_successful_attacks']),             '', ''],
        ['Attack Success Rate',    f"{adv_metrics['attack_success_rate']:.4f}",          '', ''],
        ['Flip Rate',              f"{adv_metrics['flip_rate']:.4f}",                    '', ''],
        ['Accuracy Drop',          f"{adv_metrics['accuracy_drop']:.4f}",                '', ''],
        ['Mean Conf Drop (All)',    f"{adv_metrics['mean_conf_drop']:.4f}",               '', ''],
        ['Mean Conf Drop (Succ)',   f"{adv_metrics['mean_conf_drop_successful']:.4f}",    '', ''],
        ['ASR on Vulnerable',      f"{adv_metrics['asr_on_vulnerable']:.4f}",            '', ''],
        ['ASR on Non-Vulnerable',  f"{adv_metrics['asr_on_nonvulnerable']:.4f}",         '', ''],
        ['Flip → Vulnerable',      str(adv_metrics['label_flip_to_vuln']),               '', ''],
        ['Flip → Non-Vulnerable',  str(adv_metrics['label_flip_to_nonvuln']),            '', ''],
    ]
    all_rows   = clf_rows + adv_rows
    col_labels = ['Metric', 'Original', 'Adversarial', 'Delta']
    fig_h      = 0.38 * len(all_rows) + 1.5
    fig, ax    = plt.subplots(figsize=(14, max(fig_h, 6)))
    ax.axis('off')
    tbl = ax.table(cellText=all_rows, colLabels=col_labels,
                   cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    split = len(clf_rows)
    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif row <= split:
            cell.set_facecolor('#d6eaf8' if row % 2 == 0 else '#ffffff')
        else:
            cell.set_facecolor('#fdfefe' if row % 2 == 0 else '#f9ebea')
        cell.set_edgecolor('#bdc3c7')
        if col == 3 and 0 < row <= split:
            try:
                val = float(all_rows[row - 1][3])
                if val < 0:
                    cell.set_text_props(color='red', fontweight='bold')
                elif val > 0:
                    cell.set_text_props(color='green', fontweight='bold')
            except Exception:
                pass
    ax.set_title('LAPPEFT — Adversarial Analysis Full Summary Table', fontweight='bold', fontsize=12, pad=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")
def print_full_results(adv_metrics, orig_clf_m, adv_clf_m):
    scalar_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision',
                   'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity', 'FPR', 'FNR']
    sep = "=" * 70
    print(f"\n{sep}")
    print("  ADVERSARIAL ROBUSTNESS ANALYSIS")
    print(sep)
    print(f"  Total samples              : {adv_metrics['n_total']}")
    print(f"  Originally correct         : {adv_metrics['n_orig_correct']}")
    print(f"  Robust correct (adv)       : {adv_metrics['n_adv_correct']}")
    print(f"  Predictions flipped        : {adv_metrics['n_flipped']}")
    print(f"  Successful attacks         : {adv_metrics['n_successful_attacks']}")
    print(f"  Attack Success Rate (ASR)  : {adv_metrics['attack_success_rate']:.4f}")
    print(f"  Flip Rate                  : {adv_metrics['flip_rate']:.4f}")
    print(f"  Original Accuracy          : {adv_metrics['original_accuracy']:.4f}")
    print(f"  Robust Accuracy            : {adv_metrics['robust_accuracy']:.4f}")
    print(f"  Accuracy Drop              : {adv_metrics['accuracy_drop']:.4f}")
    print(f"  Mean Conf Drop (all)       : {adv_metrics['mean_conf_drop']:.4f}")
    print(f"  Mean Conf Drop (successful): {adv_metrics['mean_conf_drop_successful']:.4f}")
    print(f"  Orig Conf (correct preds)  : {adv_metrics['orig_conf_on_correct']:.4f}")
    print(f"  Adv  Conf (correct preds)  : {adv_metrics['adv_conf_on_correct']:.4f}")
    print(f"  ASR on Vulnerable          : {adv_metrics['asr_on_vulnerable']:.4f}")
    print(f"  ASR on Non-Vulnerable      : {adv_metrics['asr_on_nonvulnerable']:.4f}")
    print(f"  Flip → Vulnerable          : {adv_metrics['label_flip_to_vuln']}")
    print(f"  Flip → Non-Vulnerable      : {adv_metrics['label_flip_to_nonvuln']}")
    print(f"\n{sep}")
    print("  CLASSIFICATION METRICS: ORIGINAL vs ADVERSARIAL CODE")
    print(sep)
    print(f"  {'Metric':<22} {'Original':>12} {'Adversarial':>12} {'Delta':>10}")
    print(f"  {'-'*58}")
    for k in scalar_keys:
        o = orig_clf_m[k]
        a = adv_clf_m[k]
        print(f"  {k:<22} {o:>12.4f} {a:>12.4f} {o - a:>+10.4f}")
    print(f"  {'TP':<22} {orig_clf_m['TP']:>12}  {adv_clf_m['TP']:>11}")
    print(f"  {'TN':<22} {orig_clf_m['TN']:>12}  {adv_clf_m['TN']:>11}")
    print(f"  {'FP':<22} {orig_clf_m['FP']:>12}  {adv_clf_m['FP']:>11}")
    print(f"  {'FN':<22} {orig_clf_m['FN']:>12}  {adv_clf_m['FN']:>11}")
    for section, data_dict in [
        ('PER ATTACK METHOD',         adv_metrics['per_attack_method']),
        ('PER VULNERABILITY CATEGORY', adv_metrics['per_category']),
        ('PER CWE',                   adv_metrics['per_cwe']),
    ]:
        print(f"\n{sep}")
        print(f"  {section}")
        print(sep)
        for key, vals in data_dict.items():
            print(f"  [{key}]  n={vals['n']}  ASR={vals['asr']:.4f}  "
                  f"Flip={vals['flip_rate']:.4f}  RobAcc={vals['robust_acc']:.4f}  "
                  f"OrigAcc={vals['orig_acc']:.4f}  ConfDrop={vals['conf_drop']:.4f}  "
                  f"Successful={vals['n_successful']}")
    print(f"\n{sep}")
    print("  PER ATTACK SUCCESS TYPE")
    print(sep)
    for key, vals in adv_metrics['per_attack_success_type'].items():
        print(f"  [{key}]  n={vals['n']}  SuccessRate={vals['success_rate']:.4f}  "
              f"FlipRate={vals['flip_rate']:.4f}  ConfDrop={vals['conf_drop']:.4f}  "
              f"n_success={vals['n_success']}")
    print(sep)
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"Device     : {DEVICE}")
    print(f"Output dir : {OUT_DIR}")
    print(f"\nLoading tokenizer  : {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    print(f"Loading model arch : {MODEL_NAME}")
    model      = LAPPEFT().to(DEVICE)
    print(f"Loading weights    : {MODEL_PATH}")
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    elif isinstance(checkpoint, dict) and any(k.startswith('encoder') for k in checkpoint.keys()):
        model.load_state_dict(checkpoint)
    else:
        model.load_state_dict(checkpoint)
    print("Model loaded.\n")
    sep = "=" * 70
    print(sep)
    print(f"  Loading adversarial CSV: {ADV_CSV}")
    adv_df = pd.read_csv(ADV_CSV)
    print(f"  Raw rows        : {len(adv_df)}")
    print(f"  Columns present : {list(adv_df.columns)}")
    for col in ['original_code', 'adversarial_code', 'vuln_label']:
        if col not in adv_df.columns:
            raise ValueError(f"Required column '{col}' not found. Available: {list(adv_df.columns)}")
    adv_df = adv_df.dropna(subset=['original_code', 'adversarial_code', 'vuln_label']).reset_index(drop=True)
    print(f"  After dropna    : {len(adv_df)}")
    print(f"  Label dist      : {dict(adv_df['vuln_label'].value_counts().sort_index())}")
    for col in ['attack_method', 'attack_success_type', 'category', 'cwe']:
        if col in adv_df.columns:
            print(f"  {col:<25}: {adv_df[col].value_counts().to_dict()}")
    print(sep)
    orig_ds = RawCodeDataset(adv_df['original_code'].tolist(),    adv_df['vuln_label'].tolist(), tokenizer)
    adv_ds  = RawCodeDataset(adv_df['adversarial_code'].tolist(), adv_df['vuln_label'].tolist(), tokenizer)
    pin_mem = DEVICE.type == 'cuda'
    orig_loader = DataLoader(orig_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=4, pin_memory=pin_mem)
    adv_loader  = DataLoader(adv_ds,  batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=4, pin_memory=pin_mem)
    print("\nRunning inference on original_code column ...")
    orig_labels, orig_preds, orig_probs = run_inference(model, orig_loader)
    print("Running inference on adversarial_code column ...")
    adv_labels,  adv_preds,  adv_probs  = run_inference(model, adv_loader)
    orig_clf_m  = compute_clf_metrics(orig_labels, orig_preds, orig_probs)
    adv_clf_m   = compute_clf_metrics(adv_labels,  adv_preds,  adv_probs)
    adv_metrics = compute_adversarial_metrics(
        orig_labels, orig_preds, orig_probs,
        adv_preds,   adv_probs,  adv_df
    )
    print_full_results(adv_metrics, orig_clf_m, adv_clf_m)
    print(f"\n{sep}")
    print("  Generating figures ...")
    print(sep)
    plot_adv_overview(adv_metrics,
                      os.path.join(OUT_DIR, '01_adv_overview.png'))
    plot_orig_vs_adv_cm(orig_labels, orig_preds, adv_preds,
                         os.path.join(OUT_DIR, '02_orig_vs_adv_cm.png'))
    plot_radar_orig_vs_adv(orig_clf_m, adv_clf_m,
                            os.path.join(OUT_DIR, '03_radar_orig_vs_adv.png'))
    plot_metric_drop_bar(orig_clf_m, adv_clf_m,
                          os.path.join(OUT_DIR, '04_metric_drop_bar.png'))
    plot_conf_shift(orig_probs, adv_probs, orig_labels,
                     os.path.join(OUT_DIR, '05_conf_shift_distribution.png'))
    plot_metrics_heatmap_orig_vs_adv(orig_clf_m, adv_clf_m,
                                      os.path.join(OUT_DIR, '06_metrics_heatmap_orig_vs_adv.png'))
    plot_asr_label_class(adv_metrics,
                          os.path.join(OUT_DIR, '07_asr_by_label_class.png'))
    plot_per_group(adv_metrics['per_attack_method'],  'Attack Method',
                   'Attack Method', os.path.join(OUT_DIR, '08_per_attack_method.png'))
    plot_per_group(adv_metrics['per_category'],        'category',
                   'Vulnerability Category', os.path.join(OUT_DIR, '09_per_category.png'))
    plot_per_group(adv_metrics['per_cwe'],             'cwe',
                   'CWE', os.path.join(OUT_DIR, '10_per_cwe.png'))
    plot_attack_success_type(adv_metrics['per_attack_success_type'],
                              os.path.join(OUT_DIR, '11_attack_success_type.png'))
    plot_heatmap_method_cwe(adv_metrics['per_attack_method'], adv_metrics['per_cwe'],
                             os.path.join(OUT_DIR, '12_heatmap_method_cwe.png'))
    plot_conf_asr_per_method_scatter(adv_metrics['per_attack_method'],
                                      os.path.join(OUT_DIR, '13_asr_vs_conf_drop_scatter.png'))
    plot_summary_table(adv_metrics, orig_clf_m, adv_clf_m,
                        os.path.join(OUT_DIR, '14_summary_table.png'))
    print(f"\n{sep}")
    print("  Saving CSVs ...")
    print(sep)
    scalar_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision',
                   'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity', 'FPR', 'FNR']
    clf_rows = [{'Metric': k, 'Original': round(orig_clf_m[k], 4),
                 'Adversarial': round(adv_clf_m[k], 4),
                 'Delta': round(orig_clf_m[k] - adv_clf_m[k], 4)} for k in scalar_keys]
    pd.DataFrame(clf_rows).to_csv(
        os.path.join(OUT_DIR, 'clf_metrics_orig_vs_adv.csv'), index=False)
    print(f"  Saved: clf_metrics_orig_vs_adv.csv")
    rob_summary = {k: v for k, v in adv_metrics.items()
                   if not isinstance(v, dict)}
    pd.DataFrame([rob_summary]).to_csv(
        os.path.join(OUT_DIR, 'robustness_summary.csv'), index=False)
    print(f"  Saved: robustness_summary.csv")
    for fname, data_dict, id_col in [
        ('per_attack_method.csv',        adv_metrics['per_attack_method'],        'attack_method'),
        ('per_category.csv',             adv_metrics['per_category'],             'category'),
        ('per_cwe.csv',                  adv_metrics['per_cwe'],                  'cwe'),
        ('per_attack_success_type.csv',  adv_metrics['per_attack_success_type'],  'attack_success_type'),
    ]:
        if data_dict:
            rows = [{id_col: k, **{mk: round(v, 4) if isinstance(v, float) else v
                                   for mk, v in vals.items()}}
                    for k, vals in data_dict.items()]
            pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, fname), index=False)
            print(f"  Saved: {fname}")
    sample_df = adv_df.copy()
    sample_df['orig_pred']         = orig_preds
    sample_df['adv_pred']          = adv_preds
    sample_df['orig_prob_vuln']    = np.round(orig_probs, 4)
    sample_df['adv_prob_vuln']     = np.round(adv_probs, 4)
    sample_df['conf_drop']         = np.round(orig_probs - adv_probs, 4)
    sample_df['pred_flipped']      = (orig_preds != adv_preds).astype(int)
    sample_df['orig_correct']      = (orig_preds == adv_df['vuln_label'].values).astype(int)
    sample_df['adv_correct']       = (adv_preds  == adv_df['vuln_label'].values).astype(int)
    sample_df['attack_successful'] = (
        sample_df['orig_correct'].astype(bool) & sample_df['pred_flipped'].astype(bool)
    ).astype(int)
    sample_df.to_csv(os.path.join(OUT_DIR, 'sample_level_results.csv'), index=False)
    print(f"  Saved: sample_level_results.csv")
    print(f"\n{sep}")
    print(f"  All outputs saved to: {OUT_DIR}")
    print(f"  Files generated:")
    for fname in sorted(os.listdir(OUT_DIR)):
        print(f"    {fname}")
    print(sep)
    print("\nDone.")
if __name__ == "__main__":
    main()

# Hieradapter Inference on in-domain-distribution dataset for code vulnerability (UniXcoder)

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, matthews_corrcoef, confusion_matrix,
    balanced_accuracy_score, average_precision_score
)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

MODEL_NAME  = "microsoft/unixcoder-base"
MODEL_PATH  = "/best_proposed_model.pt"
MAX_LENGTH  = 512
D_MODEL     = 768
BATCH_SIZE  = 16
BOTTLENECK  = 6
NUM_LEVELS  = 4
LAYERS_LEVEL = 3
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

test_configs = [
    {'csv': 't/big_vuln.csv',     'label_col': 'label', 'code_col': 'func',  'name': 'BIG_VULTEST'},
    {'csv': 't/mix_vuln.csv',     'label_col': 'label', 'code_col': 'func',  'name': 'MIX_VULTEST'},
    {'csv': 't/revealn.csv',      'label_col': 'label', 'code_col': 'func',  'name': 'REVEAL_VULTEST'},
    {'csv': 't/julietn.csv',      'label_col': 'label', 'code_col': 'func',  'name': 'JULIET_VULTEST'},
    {'csv': 't/diversen.csv',     'label_col': 'label', 'code_col': 'func',  'name': 'DIVERSE_VULTEST'},
    {'csv': 't/sven.csv',         'label_col': 'label', 'code_col': 'func',  'name': 'SVEN'},
    {'csv': 't/primetestn.csv',   'label_col': 'label', 'code_col': 'func',  'name': 'Prime'},
    {'csv': 'testcodex.csv',      'label_col': 'label', 'code_col': 'code',  'name': 'CDL_TEST'},
]


class CodeDataset(Dataset):
    def __init__(self, df, tokenizer, code_col, label_col):
        self.codes     = df[code_col].astype(str).tolist()
        self.labels    = df[label_col].tolist()
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.codes)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.codes[idx],
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }


class AttentionPooling(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.attn = nn.Linear(d_model, 1)

    def forward(self, hidden, attention_mask):
        scores  = self.attn(hidden).squeeze(-1)
        scores  = scores.masked_fill(attention_mask == 0, -1e9)
        weights = torch.softmax(scores, dim=-1)
        return (hidden * weights.unsqueeze(-1)).sum(1)


class DynamicAlphaFusion(nn.Module):
    def __init__(self, d_model, num_levels):
        super().__init__()
        self.gate = nn.Linear(d_model, num_levels)

    def forward(self, base_cls, residuals):
        alphas   = torch.softmax(self.gate(base_cls), dim=-1)
        weighted = sum(alphas[:, i:i+1] * residuals[i] for i in range(len(residuals)))
        return base_cls + weighted


class LaplacianResidualAdapter(nn.Module):
    def __init__(self, d_model, bottleneck_dim):
        super().__init__()
        self.down    = nn.Linear(d_model, bottleneck_dim)
        self.act     = nn.GELU()
        self.up      = nn.Linear(bottleneck_dim, d_model)
        self.dropout = nn.Dropout(0.1)
        nn.init.kaiming_normal_(self.down.weight, nonlinearity='linear')
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return self.up(self.dropout(self.act(self.down(x))))


class LAPPEFT(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(MODEL_NAME)
        for p in self.encoder.parameters():
            p.requires_grad = False

        self.pool_layers = nn.ModuleList([
            AttentionPooling(D_MODEL) for _ in range(NUM_LEVELS + 1)
        ])

        self.lras = nn.ModuleList([
            LaplacianResidualAdapter(D_MODEL, BOTTLENECK)
            for _ in range(NUM_LEVELS)
        ])

        self.level_norms = nn.ModuleList([nn.LayerNorm(D_MODEL) for _ in range(NUM_LEVELS)])

        self.fusion = DynamicAlphaFusion(D_MODEL, NUM_LEVELS)

        self.classifier = nn.Sequential(
            nn.Linear(D_MODEL, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(64, 2)
        )

    def encode(self, input_ids, attention_mask):
        out        = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True
        )
        hidden     = out.hidden_states
        num_layers = len(hidden)

        base = self.pool_layers[NUM_LEVELS](hidden[-1], attention_mask)

        level_indices = [
            min((lvl + 1) * LAYERS_LEVEL, num_layers - 1) for lvl in range(NUM_LEVELS)
        ]

        level_cls = [
            self.pool_layers[lvl](hidden[level_indices[lvl]], attention_mask)
            for lvl in range(NUM_LEVELS)
        ]

        residuals = []
        for lvl in range(NUM_LEVELS):
            diff = level_cls[lvl] - level_cls[lvl + 1] if lvl < NUM_LEVELS - 1 else level_cls[lvl]
            r    = self.lras[lvl](diff)
            residuals.append(self.level_norms[lvl](r))

        recon = self.fusion(base, residuals)
        return recon

    def forward(self, input_ids, attention_mask, cls_embed=None):
        if cls_embed is not None:
            return self.classifier(cls_embed)
        recon = self.encode(input_ids, attention_mask)
        return self.classifier(recon)


def compute_metrics(labels, preds, probs):
    labels = np.array(labels)
    preds  = np.array(preds)
    probs  = np.array(probs)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    return {
        'Accuracy':      accuracy_score(labels, preds),
        'Balanced_Acc':  balanced_accuracy_score(labels, preds),
        'Precision':     precision_score(labels, preds, zero_division=0),
        'Recall':        recall_score(labels, preds, zero_division=0),
        'F1':            f1_score(labels, preds, zero_division=0),
        'ROC_AUC':       roc_auc_score(labels, probs),
        'Avg_Precision': average_precision_score(labels, probs),
        'MCC':           matthews_corrcoef(labels, preds),
        'Specificity':   tn / (tn + fp + 1e-9),
        'Sensitivity':   tp / (tp + fn + 1e-9),
        'FPR':           fp / (fp + tn + 1e-9),
        'FNR':           fn / (fn + tp + 1e-9),
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }


def run_inference(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['label'].to(DEVICE)
            logits         = model(input_ids, attention_mask)
            probs          = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            preds          = logits.argmax(dim=-1).cpu().numpy()
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds)
            all_probs.extend(probs)
    return all_labels, all_preds, all_probs


def plot_confusion_matrix(m, title, save_path):
    cm      = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
    fig, ax = plt.subplots(figsize=(5, 4))
    im      = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred 0', 'Pred 1'])
    ax.set_yticklabels(['True 0', 'True 1'])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > thresh else 'black',
                    fontsize=14, fontweight='bold')
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def plot_per_dataset_metrics(all_results, save_path='per_dataset_metrics.png'):
    datasets     = [r['name'] for r in all_results]
    metric_keys  = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity']
    colors       = plt.cm.tab10(np.linspace(0, 1, len(metric_keys)))

    fig, axes = plt.subplots(2, 5, figsize=(26, 10))
    fig.suptitle(' Per-Dataset Binary Classification Metrics', fontsize=15, fontweight='bold')
    axes = axes.flatten()

    for idx, key in enumerate(metric_keys):
        vals = [r['metrics'][key] for r in all_results]
        bars = axes[idx].bar(datasets, vals, color=colors[idx], alpha=0.85, edgecolor='black', linewidth=0.5)
        axes[idx].set_title(key, fontweight='bold', fontsize=11)
        axes[idx].set_ylim(0, 1.12)
        axes[idx].set_ylabel('Score')
        axes[idx].tick_params(axis='x', rotation=40, labelsize=7)
        axes[idx].grid(axis='y', alpha=0.3)
        for bar in bars:
            axes[idx].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                           f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=6.5, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Per-dataset metrics chart saved to {save_path}")


def plot_heatmap(all_results, save_path='metrics_heatmap.png'):
    metric_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity', 'FPR', 'FNR']
    datasets    = [r['name'] for r in all_results]
    data        = np.array([[r['metrics'][k] for k in metric_keys] for r in all_results])

    fig, ax = plt.subplots(figsize=(18, 6))
    im      = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    plt.colorbar(im, ax=ax, fraction=0.02, pad=0.04)
    ax.set_xticks(range(len(metric_keys)))
    ax.set_yticks(range(len(datasets)))
    ax.set_xticklabels(metric_keys, rotation=35, ha='right', fontsize=10)
    ax.set_yticklabels(datasets, fontsize=10)
    ax.set_title('LAPPEFT — Inference Metrics Heatmap Across Datasets', fontsize=13, fontweight='bold')

    for i in range(len(datasets)):
        for j in range(len(metric_keys)):
            ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center',
                    fontsize=7.5, fontweight='bold',
                    color='black' if 0.25 < data[i, j] < 0.75 else 'white')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Metrics heatmap saved to {save_path}")


def plot_radar_all(all_results, save_path='radar_all_datasets.png'):
    keys   = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Specificity', 'Sensitivity']
    n      = len(keys)
    angles = [x / float(n) * 2 * np.pi for x in range(n)]
    angles += angles[:1]

    colors = plt.cm.tab10(np.linspace(0, 1, len(all_results)))
    fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))
    ax.set_title('LAPPEFT — Radar: All Inference Datasets', fontweight='bold', pad=25, fontsize=13)

    for idx, r in enumerate(all_results):
        vals  = [r['metrics'][k] for k in keys]
        vals += vals[:1]
        ax.plot(angles, vals, color=colors[idx], linewidth=2, label=r['name'])
        ax.fill(angles, vals, color=colors[idx], alpha=0.07)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(keys, size=9)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.15), fontsize=8)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Radar chart saved to {save_path}")


def plot_grouped_bar_comparison(all_results, save_path='grouped_bar_comparison.png'):
    core_keys = ['Accuracy', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall']
    datasets  = [r['name'] for r in all_results]
    n_metrics = len(core_keys)
    n_datasets= len(datasets)
    x         = np.arange(n_metrics)
    width     = 0.85 / n_datasets
    colors    = plt.cm.tab10(np.linspace(0, 1, n_datasets))

    fig, ax = plt.subplots(figsize=(16, 7))
    for i, r in enumerate(all_results):
        vals  = [r['metrics'][k] for k in core_keys]
        offsets = x + (i - n_datasets / 2) * width + width / 2
        bars  = ax.bar(offsets, vals, width=width * 0.92, color=colors[i], alpha=0.88,
                       edgecolor='black', linewidth=0.4, label=r['name'])

    ax.set_xticks(x)
    ax.set_xticklabels(core_keys, fontsize=11)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score', fontsize=11)
    ax.set_title('LAPPEFT — Core Metrics Grouped Comparison Across All Datasets', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8, ncol=2)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Grouped bar comparison saved to {save_path}")


def plot_fpr_fnr_tradeoff(all_results, save_path='fpr_fnr_tradeoff.png'):
    fig, ax = plt.subplots(figsize=(9, 7))
    colors  = plt.cm.tab10(np.linspace(0, 1, len(all_results)))
    for idx, r in enumerate(all_results):
        ax.scatter(r['metrics']['FPR'], r['metrics']['FNR'], s=160, color=colors[idx],
                   edgecolors='black', linewidths=0.8, zorder=5, label=r['name'])
        ax.annotate(r['name'], (r['metrics']['FPR'], r['metrics']['FNR']),
                    textcoords='offset points', xytext=(8, 4), fontsize=7.5)

    ax.set_xlabel('False Positive Rate (FPR)', fontsize=11)
    ax.set_ylabel('False Negative Rate (FNR)', fontsize=11)
    ax.set_title('LAPPEFT — FPR vs FNR Trade-off Across Datasets', fontsize=13, fontweight='bold')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"FPR vs FNR trade-off plot saved to {save_path}")


def plot_confusion_matrices_grid(all_results, save_path='confusion_matrices_grid.png'):
    n       = len(all_results)
    ncols   = 4
    nrows   = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.5 * nrows))
    axes    = axes.flatten()
    fig.suptitle('Confusion Matrices for All Inference Datasets', fontsize=14, fontweight='bold')

    for idx, r in enumerate(all_results):
        m  = r['metrics']
        cm = np.array([[m['TN'], m['FP']], [m['FN'], m['TP']]])
        ax = axes[idx]
        im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xticklabels(['Pred 0', 'Pred 1'], fontsize=9)
        ax.set_yticklabels(['True 0', 'True 1'], fontsize=9)
        thresh = cm.max() / 2.0
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                        color='white' if cm[i, j] > thresh else 'black',
                        fontsize=13, fontweight='bold')
        ax.set_title(r['name'], fontweight='bold', fontsize=10)
        ax.set_xlabel('Predicted')
        ax.set_ylabel('Actual')

    for idx in range(n, len(axes)):
        axes[idx].set_visible(False)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrices grid saved to {save_path}")


def plot_mcc_f1_scatter(all_results, save_path='mcc_f1_scatter.png'):
    fig, ax = plt.subplots(figsize=(9, 7))
    colors  = plt.cm.tab10(np.linspace(0, 1, len(all_results)))
    for idx, r in enumerate(all_results):
        ax.scatter(r['metrics']['MCC'], r['metrics']['F1'], s=180, color=colors[idx],
                   edgecolors='black', linewidths=0.8, zorder=5, label=r['name'])
        ax.annotate(r['name'], (r['metrics']['MCC'], r['metrics']['F1']),
                    textcoords='offset points', xytext=(8, 4), fontsize=7.5)

    ax.set_xlabel('MCC', fontsize=11)
    ax.set_ylabel('F1 Score', fontsize=11)
    ax.set_title('LAPPEFT — MCC vs F1 Score Across Datasets', fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"MCC vs F1 scatter plot saved to {save_path}")


def plot_summary_table(all_results, save_path='summary_table.png'):
    metric_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision', 'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity', 'FPR', 'FNR']
    datasets    = [r['name'] for r in all_results]
    data        = [[f"{r['metrics'][k]:.4f}" for k in metric_keys] for r in all_results]

    col_widths  = [max(len(m) for m in metric_keys)] + [max(len(d), max(len(row[i]) for row in data)) for i, d in enumerate(metric_keys)]
    fig_w       = 1.1 * len(metric_keys)
    fig_h       = 0.5 * (len(datasets) + 2)
    fig, ax     = plt.subplots(figsize=(max(fig_w, 20), max(fig_h, 4)))
    ax.axis('off')

    table_data  = [metric_keys] + [[r['name']] + [f"{r['metrics'][k]:.4f}" for k in metric_keys] for r in all_results]
    col_labels  = ['Dataset'] + metric_keys
    row_data    = [[r['name']] + [f"{r['metrics'][k]:.4f}" for k in metric_keys] for r in all_results]

    tbl = ax.table(cellText=row_data, colLabels=col_labels,
                   cellLoc='center', loc='center',
                   bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7.5)

    for (row, col), cell in tbl.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif row % 2 == 0:
            cell.set_facecolor('#ecf0f1')
        else:
            cell.set_facecolor('#ffffff')
        cell.set_edgecolor('#bdc3c7')

    ax.set_title('LAPPEFT — Full Inference Metrics Summary Table', fontweight='bold', fontsize=12, pad=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Summary table saved to {save_path}")


def main():
    print(f"Device: {DEVICE}")
    print(f"Loading tokenizer from: {MODEL_NAME}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    print(f"Loading model architecture...")
    model = LAPPEFT().to(DEVICE)

    print(f"Loading weights from: {MODEL_PATH}")
    checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
    if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
        model.load_state_dict(checkpoint['model_state_dict'])
    elif isinstance(checkpoint, dict) and any(k.startswith('encoder') for k in checkpoint.keys()):
        model.load_state_dict(checkpoint)
    else:
        model.load_state_dict(checkpoint)
    print("Model loaded successfully.\n")

    all_results = []
    scalar_keys = ['Accuracy', 'Balanced_Acc', 'F1', 'ROC_AUC', 'MCC', 'Precision',
                   'Recall', 'Avg_Precision', 'Specificity', 'Sensitivity', 'FPR', 'FNR']

    sep = "=" * 70

    for cfg in test_configs:
        print(sep)
        print(f"  Dataset : {cfg['name']}")
        print(f"  CSV     : {cfg['csv']}")

        df = pd.read_csv(cfg['csv'])
        print(f"  Samples : {len(df)}  |  Label dist: {dict(df[cfg['label_col']].value_counts().sort_index())}")

        ds     = CodeDataset(df.reset_index(drop=True), tokenizer, cfg['code_col'], cfg['label_col'])
        loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

        labels, preds, probs = run_inference(model, loader)
        metrics              = compute_metrics(labels, preds, probs)

        print(f"\n  {'Metric':<22} {'Value':>10}")
        print(f"  {'-'*34}")
        for k in scalar_keys:
            print(f"  {k:<22} {metrics[k]:>10.4f}")
        print(f"  {'TP':<22} {metrics['TP']:>10}")
        print(f"  {'TN':<22} {metrics['TN']:>10}")
        print(f"  {'FP':<22} {metrics['FP']:>10}")
        print(f"  {'FN':<22} {metrics['FN']:>10}")

        all_results.append({'name': cfg['name'], 'metrics': metrics})

    print(sep)
    print("\nGenerating figures...\n")

    plot_per_dataset_metrics(all_results,    save_path='per_dataset_metrics.png')
    plot_heatmap(all_results,                save_path='metrics_heatmap.png')
    plot_radar_all(all_results,              save_path='radar_all_datasets.png')
    plot_grouped_bar_comparison(all_results, save_path='grouped_bar_comparison.png')
    plot_fpr_fnr_tradeoff(all_results,       save_path='fpr_fnr_tradeoff.png')
    plot_confusion_matrices_grid(all_results,save_path='confusion_matrices_grid.png')
    plot_mcc_f1_scatter(all_results,         save_path='mcc_f1_scatter.png')
    plot_summary_table(all_results,          save_path='summary_table.png')

    print("\nAll figures saved.")

    rows = []
    for r in all_results:
        row = {'Dataset': r['name']}
        row.update({k: round(r['metrics'][k], 4) for k in scalar_keys})
        row.update({k: r['metrics'][k] for k in ['TP', 'TN', 'FP', 'FN']})
        rows.append(row)
    results_df = pd.DataFrame(rows)
    results_df.to_csv('inference_results_all_datasets.csv', index=False)
    print("Full results CSV saved to: inference_results_all_datasets.csv")

    print(f"\n{sep}")
    print("  AGGREGATE SUMMARY ACROSS ALL DATASETS")
    print(f"{sep}")
    print(f"  {'Metric':<22} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
    print(f"  {'-'*54}")
    for k in scalar_keys:
        vals = [r['metrics'][k] for r in all_results]
        print(f"  {k:<22} {np.mean(vals):>10.4f} {np.std(vals):>10.4f} {np.min(vals):>10.4f} {np.max(vals):>10.4f}")
    print(sep)


if __name__ == "__main__":
    main()